# DiaFuse-PRC — protocole tunisien / Arabizi (Colab)

Ce notebook exécute **de bout en bout** le protocole expérimental de l'article : téléchargement des modèles
(XLM-R-large, mE5-large, LaBSE) et des quatre corpus publics (TSAC, TUNIZI, NArabizi, MYC), reconstruction
des splits, générateurs $G_\text{seen}$ / $G_\text{unseen}$, extraction des vecteurs par couche, entraînement
de DiaFuse-PRC et des baselines, chaîne d'évaluation S0 → S6 avec les jeux de contrôle, puis production de
`results/results.tex` (à renvoyer pour compléter l'article).

**Avant de lancer :** `Exécution > Modifier le type d'exécution > GPU` (T4 au minimum ; A100/L4 conseillé pour `MODE="full"`),
puis `Exécution > Tout exécuter`. Toutes les étapes sont **reprenables** : si la session coupe, relancez tout,
les runs déjà terminés sont sautés (avec `USE_DRIVE=True`, `runs/` et `results/` sont sur Google Drive).

**Durée (ordre de grandeur, non mesuré sur GPU) :** `quick` ≈ quelques heures sur T4 ; `full` (10/5 seeds + tuning)
≈ une à deux journées GPU — préférez alors un serveur du labo (voir `download_models.py` pour le mode hors-ligne).

**À savoir avant d'interpréter les résultats**
- Les splits sont **reconstruits depuis les releases publiques** avec les règles de la Section 5.2 (dé-duplication,
  near-copies θ = 0.9, recouvrements avec TSAC). Les effectifs obtenus ne sont pas ceux de votre pipeline
  (1 279 / 547 / 1 020 / 1 737 / 696 / 1 938) : ils sont écrits dans `data/prep_report.json` et repris dans
  `results.tex`. Pour utiliser **vos** splits (et garder les effectifs du Tableau 1), renseignez `USE_OWN_SPLITS` :
  un dossier contenant `train_ar, val_ar, ar, tl, tz, na, naar, my, myar` (TSV `id, text, label`) ; toutes les vues
  dérivées ($G_	ext{seen}$, $G_	ext{unseen}$, perturbations, transliterate-test) sont générées à partir d'eux.
- $G_\text{seen}$ / $G_\text{unseen}$ implémentent exactement la table de conventions de l'annexe D.
- Non produits automatiquement : **B12′ (Setting B, LoRA)** ; les **audits humains** (feuilles exportées à l'étape 7,
  à remplir par des locuteurs natifs) ; **+Cross-lingual** (nécessite le corpus MADAR, sous licence → `MADAR_DIR`) ;
  **abl2-q1** (le filtre de validité n'est pas implémenté : q = 1 pour toutes les familles).

In [ ]:
# ============================ CONFIGURATION ============================
MODE = "quick"            # "quick": 3 seeds (main) / 2 (ablations), default hyper-parameters
                          # "full" : 10 / 5 seeds + tuning budget of 20 configurations (protocol of the paper)
USE_DRIVE = True          # keep runs/, results/ and audit/ on Google Drive (resumable across sessions)
DRIVE_DIR = "/content/drive/MyDrive/DiaFuse-PRC"
LOCAL_DIR = "/content/diafuse"   # features stay on the fast local disk (re-extracted after a disconnect)
ENCODERS = "xlmr,me5,labse"
LAYER_STRIDE = 2          # every second layer is cached (Appendix H)
RUN_FINETUNE = True       # B1 (XLM-R-large fine-tuned) and B11 (Shao et al.-style) - GPU required
FINETUNE_SEEDS = "0,1,2" if MODE == "quick" else "0,1,2,3,4,5,6,7,8,9"
RUN_LLM_PERTURB = False   # Table 14 rows 'lexical substitution' / 'paraphrase' (Qwen2.5-7B-Instruct, GPU >= 24 GB)
RUN_QWEN_COST = False     # Table 17 row '+ Qwen2.5-7B member' (GPU >= 24 GB)
RUN_M4 = False            # Appendix C: M = 4 (+ MARBERTv2)
MADAR_DIR = None          # folder with the MADAR corpus .tsv files -> '+Cross-lingual' ablation; None = skipped
USE_OWN_SPLITS = None     # folder with YOUR base splits (TSV: id, text, label): train_ar, val_ar, ar, tl, tz,
                          # na, naar (same ids as na), my, myar -> all derived views are generated from them
SKIP_GROUPS = ""          # e.g. "fact,abl2" to shorten a run; groups: B0 B2 B3 B4 B5 B6 B7 B8 B9 B10 B12 CEIL abl abl2 lt fact

In [ ]:
!nvidia-smi -L || echo "No GPU detected: Runtime > Change runtime type > GPU"
!pip -q install "transformers>=4.44" sentencepiece huggingface_hub umap-learn scikit-learn pandas

In [ ]:
import os, json, shutil, subprocess, sys, time, glob, zipfile
os.makedirs(LOCAL_DIR, exist_ok=True); os.chdir(LOCAL_DIR)
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    for d in ("runs", "results", "audit"):
        os.makedirs(f"{DRIVE_DIR}/{d}", exist_ok=True)
        if not os.path.islink(d):
            if os.path.isdir(d):
                shutil.rmtree(d)
            os.symlink(f"{DRIVE_DIR}/{d}", d)
else:
    for d in ("runs", "results", "audit"):
        os.makedirs(d, exist_ok=True)

def run(cmd, env=None):
    """Run a shell command, stream its output, stop the notebook on failure."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ, **(env or {})})
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"command failed: {cmd}")
print("working directory:", os.getcwd())

## Code du pipeline (écrit dans le répertoire de travail)

In [ ]:
%%writefile translit.py
"""Rule-based Arabic-script -> Latin-script (Arabizi) generators and perturbations.

G_seen  : deterministic, digit-heavy Tunisian conventions (3 7 5 8 9 2, ch, ou, i, el).
G_unseen: different rule set with digraphs and stochastic choices (a/e, h, kh, gh, q,
          sh, u/oo, ee/i, l/il, hamza dropped), seeded by the sentence so that the output
          is reproducible. G_unseen is used only to build evaluation sets.
These implement exactly the convention table of Appendix D of the paper.
"""
import hashlib
import random
import re
import unicodedata

AR_RANGE = re.compile(r"[؀-ۿݐ-ݿﭐ-﷿ﹰ-﻿]")
LAT_LETTER = re.compile(r"[A-Za-zÀ-ɏ]")
DIACRITICS = re.compile(r"[ً-ْٰـ]")  # harakat, dagger alef, tatweel

VOWEL_LETTERS = set("اوي")
SEEN = {
    "ب": "b", "ت": "t", "ث": "th", "ج": "j", "ح": "7", "خ": "5", "د": "d", "ذ": "dh",
    "ر": "r", "ز": "z", "س": "s", "ش": "ch", "ص": "s", "ض": "dh", "ط": "t", "ظ": "dh",
    "ع": "3", "غ": "8", "ف": "f", "ق": "9", "ك": "k", "ل": "l", "م": "m", "ن": "n",
    "ه": "h", "ة": "a", "ء": "2", "ئ": "2", "ؤ": "2", "ى": "a", "پ": "p", "ڤ": "v",
    "گ": "g", "ک": "k", "ی": "i", "ڨ": "g",
}
UNSEEN = dict(SEEN)
UNSEEN.update({"ح": "h", "خ": "kh", "غ": "gh", "ق": "q", "ع": "a", "ش": "sh", "ذ": "d",
               "ض": "d", "ظ": "z", "ء": "", "ئ": "", "ؤ": ""})


def script_of(text):
    ar = len(AR_RANGE.findall(text)); la = len(LAT_LETTER.findall(text))
    if ar + la == 0:
        return "none"
    if min(ar, la) / (ar + la) > 0.2:
        return "mixed"
    return "ar" if ar > la else "lat"


def _rng(word, salt):
    h = hashlib.md5((salt + word).encode("utf-8")).hexdigest()
    return random.Random(int(h[:8], 16))


def _translit_word(w, table, unseen):
    w = DIACRITICS.sub("", w)
    if not AR_RANGE.search(w):
        return w
    rng = _rng(w, "gu") if unseen else None
    out = []
    if w.startswith("ال") and len(w) > 3:
        out.append(rng.choice(["l", "il"]) if unseen else "el ")
        w = w[2:]
    prev_cons = False
    n = len(w)
    for i, ch in enumerate(w):
        nxt = w[i + 1] if i + 1 < n else ""
        if ch in "اأإآ":
            out.append("i" if ch == "إ" else "a"); prev_cons = False
        elif ch == "و":
            if i == 0 or nxt in VOWEL_LETTERS or (not prev_cons):
                out.append("w"); prev_cons = True
            else:
                out.append(rng.choice(["u", "oo"]) if unseen else "ou"); prev_cons = False
        elif ch == "ي":
            if i == 0 or nxt in VOWEL_LETTERS or (not prev_cons):
                out.append("y"); prev_cons = True
            else:
                out.append(rng.choice(["ee", "i"]) if unseen else "i"); prev_cons = False
        elif ch in table:
            lat = table[ch]
            # vowel epenthesis inside consonant runs
            if prev_cons and nxt and nxt not in VOWEL_LETTERS and lat:
                if unseen:
                    if rng.random() < 0.5:
                        out.append(rng.choice(["a", "e"]))
                else:
                    out.append("e")
            out.append(lat); prev_cons = bool(lat)
        elif AR_RANGE.match(ch):
            continue  # punctuation or rare letters in the Arabic block
        else:
            out.append(ch); prev_cons = False
    return "".join(out)


def g_seen(text):
    return " ".join(_translit_word(w, SEEN, False) for w in text.split())


def g_unseen(text):
    return " ".join(_translit_word(w, UNSEEN, True) for w in text.split())


def partial(text, gen, frac, seed=0):
    toks = text.split()
    r = random.Random(seed)
    k = round(frac * len(toks))
    idx = set(r.sample(range(len(toks)), k)) if toks else set()
    return " ".join(gen(t) if i in idx else t for i, t in enumerate(toks))


# ---------------------------------------------------------------- perturbations on Latin text
def inf_spelling(text, frac=0.2, seed=0):
    """Letter repetition and vowel variation (family INF)."""
    r = random.Random(seed); out = []
    swaps = {"ou": "u", "e": "a", "a": "e", "i": "y"}
    for t in text.split():
        if r.random() < frac and len(t) > 2:
            if r.random() < 0.5:
                t = t + t[-1] * r.randint(1, 3)
            else:
                k = r.choice(list(swaps))
                t = t.replace(k, swaps[k], 1)
        out.append(t)
    return " ".join(out)


def inf_abbrev(text, frac=0.2, seed=0):
    """Vowel deletion (consonant skeletons) of long tokens (family INF)."""
    r = random.Random(seed); out = []
    for t in text.split():
        if r.random() < frac and len(t) > 3:
            t = t[0] + re.sub(r"[aeiouy]", "", t[1:])
        out.append(t)
    return " ".join(out)


def char_noise(text, frac=0.2, seed=0):
    r = random.Random(seed); out = []
    letters = "abcdefghijklmnopqrstuvwxyz"
    for t in text.split():
        if r.random() < frac and len(t) > 2:
            i = r.randrange(len(t) - 1); op = r.choice(["swap", "del", "ins", "sub"])
            if op == "swap":
                t = t[:i] + t[i + 1] + t[i] + t[i + 2:]
            elif op == "del":
                t = t[:i] + t[i + 1:]
            elif op == "ins":
                t = t[:i] + r.choice(letters) + t[i:]
            else:
                t = t[:i] + r.choice(letters) + t[i + 1:]
        out.append(t)
    return " ".join(out)


# French insertion (family CS) on Arabic-script input, before transliteration.
CS_LEX = {
    "برشا": "trop", "باهي": "bien", "بهي": "bien", "مزيان": "beau", "مزيانة": "belle",
    "خايب": "nul", "خايبة": "nulle", "يعطيك": "merci", "نحب": "j'aime", "نحبك": "je t'aime",
    "فيلم": "film", "برنامج": "emission", "توا": "maintenant", "موش": "pas", "ماهوش": "pas",
    "ياسر": "trop", "صحة": "bravo", "عيشك": "merci", "الله": "allah",
}


def code_switch(text, frac=0.2, seed=0):
    r = random.Random(seed); out = []
    for t in text.split():
        key = DIACRITICS.sub("", t)
        if key in CS_LEX and r.random() < max(frac * 3, 0.6):
            out.append(CS_LEX[key])
        else:
            out.append(t)
    return " ".join(out)


if __name__ == "__main__":
    for s in ["الفيلم باهي برشا", "عجبني برشا", "يعطيك الصحة على البرنامج", "موش مزيان"]:
        print(s, "|", g_seen(s), "|", g_unseen(s), "|", g_seen(code_switch(s)))


# ---------------------------------------------------------------- inverse (for B0-TT)
_INV_MULTI = [("ch", "ش"), ("sh", "ش"), ("kh", "خ"), ("gh", "غ"), ("th", "ث"), ("dh", "ذ"),
              ("ou", "و"), ("oo", "و"), ("ee", "ي")]
_INV_ONE = {"3": "ع", "7": "ح", "5": "خ", "8": "غ", "9": "ق", "2": "ء", "b": "ب", "t": "ت",
            "j": "ج", "d": "د", "r": "ر", "z": "ز", "s": "س", "f": "ف", "k": "ك", "q": "ق",
            "l": "ل", "m": "م", "n": "ن", "h": "ه", "w": "و", "y": "ي", "g": "ڨ", "v": "ڤ",
            "p": "ب", "x": "كس", "c": "ك", "i": "ي", "o": "و", "u": "و"}


def latin_to_arabic(text):
    """Deterministic inverse of the G_seen rules: digits and digraphs to letters, long
    vowels to matres lectionis, short vowels dropped except word-final 'a'."""
    out_words = []
    for w in text.lower().split():
        if not re.search(r"[a-z0-9]", w) or w.isdigit():
            out_words.append(w); continue
        prefix = ""
        if w in ("el", "il", "l") :
            out_words.append("ال"); continue
        res, i = [], 0
        while i < len(w):
            two = w[i:i + 2]
            m = next((ar for lat, ar in _INV_MULTI if lat == two), None)
            if m:
                res.append(m); i += 2; continue
            ch = w[i]
            if ch == "a":
                res.append("ا" if i == 0 or i == len(w) - 1 else "")
            elif ch == "e":
                res.append("ا" if i == 0 else "")
            elif ch in _INV_ONE:
                res.append(_INV_ONE[ch])
            elif not ch.isascii():
                res.append(ch)
            i += 1
        out_words.append(prefix + "".join(res))
    # re-attach a detached article
    joined = []
    for w in out_words:
        if joined and joined[-1] == "ال":
            joined[-1] = "ال" + w
        else:
            joined.append(w)
    return " ".join(joined)

In [ ]:
%%writefile prep_data.py
"""Reconstruct the Tunisian/Arabizi protocol from the public releases.

Writes data/<set>.tsv (id, text, label) in the format of diafuse_eval.py and
data/prep_report.json with every count of Table 1 and Appendix F.
"""
import csv
import json
import os
import random
import re
import sys
import unicodedata
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import normalize

sys.path.insert(0, os.path.dirname(__file__))
from translit import (g_seen, g_unseen, script_of, partial, inf_spelling, inf_abbrev,
                      char_noise, code_switch, latin_to_arabic)

RAW = os.environ.get("RAW_DIR", "raw"); OUT = os.environ.get("DATA_DIR", "data")
THETA = float(os.environ.get("NEAR_THETA", "0.9")); SEED = 13
os.makedirs(OUT, exist_ok=True)
report = {}

URL = re.compile(r"https?://\S+|www\.\S+|@\w+")


def clean(t):
    t = unicodedata.normalize("NFC", str(t))
    t = URL.sub(" ", t)
    t = "".join(ch for ch in t if unicodedata.category(ch)[0] in "LNPZ" or ch in "'")
    t = re.sub(r"\s+", " ", t).strip()
    return t


def key(t):
    t = t.lower()
    t = "".join(ch for ch in t if unicodedata.category(ch)[0] in "LNZ")
    t = re.sub(r"(.)\1{2,}", r"\1\1", t)
    return re.sub(r"\s+", " ", t).strip()


def dedup(rows, tag):
    """rows: list of (text, label). Exact de-duplication on the normalised key;
    keys with conflicting labels are dropped entirely."""
    by = defaultdict(list)
    for t, y in rows:
        k = key(t)
        if k:
            by[k].append((t, y))
    out, conflicts = [], 0
    for k, lst in by.items():
        labs = {y for _, y in lst}
        if len(labs) > 1:
            conflicts += 1
            continue
        out.append(lst[0])
    report[f"{tag}/raw"] = len(rows); report[f"{tag}/unique"] = len(out)
    report[f"{tag}/exact_dup_removed"] = len(rows) - len(out) - conflicts
    report[f"{tag}/label_conflict_keys"] = conflicts
    return out


VEC = CountVectorizer(analyzer="char", ngram_range=(3, 3), lowercase=False)


def near_mask(cands, refs):
    """True where a candidate has a near-copy (cosine of char 3-gram counts >= THETA) in refs."""
    if not cands or not refs:
        return np.zeros(len(cands), bool)
    VEC.fit([" " + key(t) + " " for t in cands + refs])
    A = normalize(VEC.transform([" " + key(t) + " " for t in cands]))
    B = normalize(VEC.transform([" " + key(t) + " " for t in refs]))
    hit = np.zeros(len(cands), bool)
    for s in range(0, A.shape[0], 500):
        S = A[s:s + 500] @ B.T
        hit[s:s + 500] = np.asarray(S.max(axis=1).todense()).ravel() >= THETA
    return hit


def within_near(rows):
    """Remove near-copies inside a set (keep the first occurrence)."""
    texts = [t for t, _ in rows]
    if not texts:
        return rows, 0
    VEC.fit([" " + key(t) + " " for t in texts])
    A = normalize(VEC.transform([" " + key(t) + " " for t in texts]))
    keep = np.ones(len(rows), bool)
    for s in range(0, A.shape[0], 500):
        S = (A[s:s + 500] @ A.T).tocsr()
        for r in range(S.shape[0]):
            i = s + r
            if not keep[i]:
                continue
            row = S.getrow(r); js = row.indices[row.data >= THETA]
            for j in js:
                if j > i:
                    keep[j] = False
    return [r for r, k in zip(rows, keep) if k], int((~keep).sum())


def split_script(rows, tag):
    by = defaultdict(list)
    for t, y in rows:
        by[script_of(t)].append((t, y))
    report[f"{tag}/mixed"] = len(by["mixed"]); report[f"{tag}/noletters"] = len(by["none"])
    return by["ar"], by["lat"]


def write(name, rows, ids=None):
    with open(os.path.join(OUT, f"{name}.tsv"), "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f, delimiter="\t", quoting=csv.QUOTE_MINIMAL)
        w.writerow(["id", "text", "label"])
        for i, (t, y) in enumerate(rows):
            w.writerow([ids[i] if ids else f"{name}-{i}", t.replace("\t", " "), int(y)])
    report[f"final/{name}"] = len(rows)


def read_lines(p):
    with open(p, encoding="utf-8", errors="ignore") as f:
        return [l.strip() for l in f if l.strip()]


OWN_SPLITS = os.environ.get("OWN_SPLITS")
if not OWN_SPLITS:
    # ------------------------------------------------------------------ TSAC
    tsac_train = [(clean(t), 1) for t in read_lines(f"{RAW}/TSAC/train_pos.txt")] + \
                 [(clean(t), 0) for t in read_lines(f"{RAW}/TSAC/train_neg.txt")]
    tsac_test = [(clean(t), 1) for t in read_lines(f"{RAW}/TSAC/test_pos.txt")] + \
                [(clean(t), 0) for t in read_lines(f"{RAW}/TSAC/test_neg.txt")]
    test_u = dedup(tsac_test, "tsac_test")
    val_ar, test_lat = split_script(test_u, "tsac_test")
    report["tsac_test/arabic"] = len(val_ar); report["tsac_test/latin"] = len(test_lat)

    train_u = dedup(tsac_train, "tsac_train")
    test_keys = {key(t) for t, _ in test_u}
    train_u = [r for r in train_u if key(r[0]) not in test_keys]
    report["tsac_train/removed_in_test_split"] = report["tsac_train/unique"] - len(train_u)
    train_ar, train_lat = split_script(train_u, "tsac_train")
    report["tsac_train/arabic"] = len(train_ar); report["tsac_train/latin"] = len(train_lat)
    # near-copies of validation inside training (Arabic script)
    nc = near_mask([t for t, _ in train_ar], [t for t, _ in val_ar])
    train_ar = [r for r, h in zip(train_ar, nc) if not h]
    report["tsac_train/near_copy_of_val_removed"] = int(nc.sum())

    rng = random.Random(SEED)
    idx = {0: [], 1: []}
    for i, (_, y) in enumerate(train_ar):
        idx[y].append(i)
    s0 = set()
    for y, lst in idx.items():
        rng.shuffle(lst); s0 |= set(lst[: round(0.1 * len(lst))])
    S0 = [train_ar[i] for i in sorted(s0)]
    TR = [train_ar[i] for i in range(len(train_ar)) if i not in s0]

    # TSAC-Latin: human Latin-script TSAC sentences (training-split Latin + test-split Latin),
    # minus within-set near-copies and near-copies of any training/validation sentence
    tl_pool = dedup(train_lat + test_lat, "tsac_latin_pool")
    tl_pool, n_within = within_near(tl_pool)
    report["tsac_latin/near_within_removed"] = n_within
    ref_lat = [g_seen(t) for t, _ in TR + val_ar + S0] + [g_unseen(t) for t, _ in TR + val_ar + S0]
    hit = near_mask([t for t, _ in tl_pool], ref_lat)
    report["tsac_latin/cross_script_removed"] = int(hit.sum())
    TL = [r for r, h in zip(tl_pool, hit) if not h]

    # reference pool for external overlap: all TSAC in both scripts (+ transliterations)
    tsac_all = [t for t, _ in train_u + test_u]
    tsac_ref_lat = [t for t in tsac_all if script_of(t) == "lat"] + \
                   [g_seen(t) for t in tsac_all if script_of(t) == "ar"] + \
                   [g_unseen(t) for t in tsac_all if script_of(t) == "ar"]
    tsac_ref_ar = [t for t in tsac_all if script_of(t) == "ar"]


    def external(rows, tag, script="lat"):
        rows = dedup(rows, tag)
        ar, lat = split_script(rows, tag)
        rows = lat if script == "lat" else ar
        report[f"{tag}/in_script"] = len(rows)
        rows, n_within = within_near(rows)
        report[f"{tag}/near_within_removed"] = n_within
        ref = tsac_ref_lat if script == "lat" else tsac_ref_ar
        hit = near_mask([t for t, _ in rows], ref)
        report[f"{tag}/tsac_overlap_removed"] = int(hit.sum())
        return [r for r, h in zip(rows, hit) if not h]


    # ------------------------------------------------------------------ TUNIZI
    tz_rows = []
    for l in read_lines(f"{RAW}/TUNIZI-Sentiment-Analysis-Tunisian-Arabizi-Dataset/TUNIZI-Dataset.txt"):
        if ";" in l:
            lab, txt = l.split(";", 1)
            if lab.strip() in ("1", "-1"):
                tz_rows.append((clean(txt), 1 if lab.strip() == "1" else 0))
    TZ = external(tz_rows, "tunizi")

    # ------------------------------------------------------------------ NArabizi (parallel)
    na, naar = [], []
    for sp in ("train", "dev", "test"):
        a = list(csv.reader(open(f"{RAW}/Narabizi/data/sentiment/ar_na/{sp}.csv", encoding="utf-8")))
        b = list(csv.reader(open(f"{RAW}/Narabizi/data/sentiment/ar_dz/{sp}.csv", encoding="utf-8")))
        for x, y in zip(a, b):
            na.append((clean(x[1]), int(x[0]))); naar.append((clean(y[1]), int(y[0])))
    report["narabizi/raw_pairs"] = len(na)
    seen_k, pairs = set(), []
    for (t, y), (ta, _) in zip(na, naar):
        k = key(t)
        if k and k not in seen_k and script_of(t) == "lat":
            seen_k.add(k); pairs.append(((t, y), (ta, y)))
    report["narabizi/unique_latin"] = len(pairs)
    hit = near_mask([p[0][0] for p in pairs], tsac_ref_lat)
    report["narabizi/tsac_overlap_removed"] = int(hit.sum())
    pairs = [p for p, h in zip(pairs, hit) if not h]
    NA = [p[0] for p in pairs]; NAAR = [p[1] for p in pairs]

    # ------------------------------------------------------------------ MYC
    myc = pd.read_csv(f"{RAW}/MYC/DATA_CLEANED.csv", encoding="utf-16")
    report["myc/raw"] = len(myc); report["myc/polarity_values"] = sorted(map(int, myc.polarity.unique()))
    my_rows = [(clean(t), 1 if int(p) == 1 else 0) for t, p in zip(myc.sentence, myc.polarity)]
    MY = external(my_rows, "myc_lat", "lat")
    MYAR_all = external(my_rows, "myc_ar", "ar")
    report["myc_ar/after_filtering"] = len(MYAR_all)
    # control sampled to match S6 in size and class balance
    r2 = random.Random(SEED)
    n_pos = int(round(np.mean([y for _, y in MY]) * len(MY))); n_neg = len(MY) - n_pos
    pos = [r for r in MYAR_all if r[1] == 1]; neg = [r for r in MYAR_all if r[1] == 0]
    r2.shuffle(pos); r2.shuffle(neg)
    MYAR = pos[:min(n_pos, len(pos))] + neg[:min(n_neg, len(neg))]
    r2.shuffle(MYAR)

else:
    # your own base splits (id, text, label): derived views are generated exactly as for the reconstruction
    def load_base(name):
        with open(os.path.join(OWN_SPLITS, f"{name}.tsv"), encoding="utf-8", newline="") as f:
            rows = list(csv.DictReader(f, delimiter="\t"))
        return [(clean(r["text"]), int(r["label"])) for r in rows], [r["id"] for r in rows]
    TR, _ = load_base("train_ar"); val_ar, _ = load_base("val_ar"); S0, _ = load_base("ar")
    TL, _ = load_base("tl"); TZ, _ = load_base("tz"); MY, _ = load_base("my"); MYAR, _ = load_base("myar")
    NA, _ida = load_base("na"); _nb, _idb = load_base("naar"); _m = dict(zip(_idb, _nb))
    NAAR = [_m[i] for i in _ida]  # naar.tsv must use the ids of na.tsv
    report["own_splits"] = OWN_SPLITS
# ------------------------------------------------------------------ write sets
# shuffle once (fixed seed) so that files are not ordered by class
_r = random.Random(SEED)
for _lst in (TR, S0, val_ar, TL, TZ, MY):
    _r.shuffle(_lst)
_perm = list(range(len(NA))); _r.shuffle(_perm)
NA = [NA[i] for i in _perm]; NAAR = [NAAR[i] for i in _perm]
ids_s0 = [f"s0-{i}" for i in range(len(S0))]
write("train_ar", TR, [f"tr-{i}" for i in range(len(TR))])
write("train_gs", [(g_seen(t), y) for t, y in TR], [f"tr-{i}" for i in range(len(TR))])
write("val_ar", val_ar, [f"va-{i}" for i in range(len(val_ar))])
write("val_gs", [(g_seen(t), y) for t, y in val_ar], [f"va-{i}" for i in range(len(val_ar))])
write("ar", S0, ids_s0)
write("gs", [(g_seen(t), y) for t, y in S0], ids_s0)
write("gu", [(g_unseen(t), y) for t, y in S0], ids_s0)
write("tl", TL); write("tz", TZ)
ids_na = [f"na-{i}" for i in range(len(NA))]
write("na", NA, ids_na); write("naar", NAAR, ids_na)
ids_my = [f"myar-{i}" for i in range(len(MYAR))]
write("my", MY); write("myar", MYAR, ids_my)
write("gsmy", [(g_seen(t), y) for t, y in MYAR], ids_my)
write("train_gu", [(g_unseen(t), y) for t, y in TR], [f"tr-{i}" for i in range(len(TR))])
# transliterate-test inputs for B0-TT (Latin-script sets mapped back to Arabic script)
for name, rows, ids in (("gs", [(g_seen(t), y) for t, y in S0], ids_s0), ("gu", [(g_unseen(t), y) for t, y in S0], ids_s0),
                        ("tl", TL, None), ("tz", TZ, None), ("na", NA, ids_na), ("my", MY, None),
                        ("gsmy", [(g_seen(t), y) for t, y in MYAR], ids_my)):
    write("tt-" + name, [(latin_to_arabic(t), y) for t, y in rows], ids if ids else [f"{name}-{i}" for i in range(len(rows))])
# training-time variants (consistency) for the Latin view
write("train_inf", [(inf_spelling(g_seen(t), 0.2, i), y) for i, (t, y) in enumerate(TR)], [f"tr-{i}" for i in range(len(TR))])
write("train_noise", [(char_noise(g_seen(t), 0.2, i), y) for i, (t, y) in enumerate(TR)], [f"tr-{i}" for i in range(len(TR))])
write("train_cs", [(g_seen(code_switch(t, 0.2, i)), y) for i, (t, y) in enumerate(TR)], [f"tr-{i}" for i in range(len(TR))])
# robustness sets at severity 2 (20% of tokens) and the severity curve for G_unseen
gs_s0 = [(g_seen(t), y) for t, y in S0]
write("rob-trs", [(partial(t, g_seen, 0.2, i), y) for i, (t, y) in enumerate(S0)], ids_s0)
write("rob-tru", [(partial(t, g_unseen, 0.2, i), y) for i, (t, y) in enumerate(S0)], ids_s0)
write("rob-spell", [(inf_spelling(t, 0.2, i), y) for i, (t, y) in enumerate(gs_s0)], ids_s0)
write("rob-abbr", [(inf_abbrev(t, 0.2, i), y) for i, (t, y) in enumerate(gs_s0)], ids_s0)
write("rob-cs", [(g_seen(code_switch(t, 0.2, i)), y) for i, (t, y) in enumerate(S0)], ids_s0)
write("rob-noise", [(char_noise(t, 0.2, i), y) for i, (t, y) in enumerate(gs_s0)], ids_s0)
for k, frac in enumerate([0.0, 0.1, 0.2, 0.35, 0.5]):
    write(f"rob-tru-s{k}", [(partial(t, g_unseen, frac, i), y) for i, (t, y) in enumerate(S0)], ids_s0)

# optional MADAR Tunis/Sfax-MSA pairs for the +Cross-lingual ablation (licence: request from the MADAR authors)
MADAR_DIR = os.environ.get("MADAR_DIR")
if MADAR_DIR and os.path.isdir(MADAR_DIR):
    by_sid = defaultdict(dict)
    for root, _, files in os.walk(MADAR_DIR):
        for fn in files:
            if not fn.endswith(".tsv"):
                continue
            df = pd.read_csv(os.path.join(root, fn), sep="\t", quoting=3, dtype=str)
            cols = {c.lower(): c for c in df.columns}
            lc = next((cols[c] for c in cols if c.startswith("lang")), None)
            sc_ = next((cols[c] for c in cols if c.startswith("sent") and "id" not in c), None)
            ic = next((cols[c] for c in cols if "id" in c), None)
            spc = next((cols[c] for c in cols if c.startswith("split")), None)
            if not (lc and sc_ and ic):
                continue
            for _, r in df.iterrows():
                if spc and str(r[spc]).lower().startswith(("dev", "test")):
                    continue
                by_sid[str(r[ic])][str(r[lc]).upper()] = clean(r[sc_])
    pairs = [(d[c], d["MSA"]) for d in by_sid.values() for c in ("TUN", "SFX") if c in d and "MSA" in d]
    report["madar/pairs"] = len(pairs)
    ids_m = [f"madar-{i}" for i in range(len(pairs))]
    write("train_madar_tn", [(a, 0) for a, _ in pairs], ids_m)
    write("train_madar_msa", [(b, 0) for _, b in pairs], ids_m)

for name, rows in (("train_ar", TR), ("ar", S0), ("tl", TL), ("tz", TZ), ("na", NA), ("my", MY), ("myar", MYAR)):
    report[f"pos_share/{name}"] = round(float(np.mean([y for _, y in rows])), 4)
json.dump(report, open(os.path.join(OUT, "prep_report.json"), "w"), indent=1, ensure_ascii=False)
# counts for Table 1 / Appendix F of the paper (read by diafuse_eval.py --manual)
if not OWN_SPLITS:
    fi = lambda n: f"{int(n):,}"
    R = report
    manual = {
        "data/train/n0": fi(R["tsac_train/arabic"]),
        "data/train/rm": fi(R["tsac_train/near_copy_of_val_removed"] + len(S0)),
        "data/val/n0": fi(R["tsac_test/unique"]), "data/val/rm": fi(R["tsac_test/unique"] - len(val_ar)),
        "data/val/latin": fi(R["tsac_test/latin"]), "data/val/n": fi(len(val_ar)),
        "data/tl/n0": fi(R["tsac_latin_pool/unique"]),
        "data/tl/rm": fi(R["tsac_latin/near_within_removed"] + R["tsac_latin/cross_script_removed"]),
        "data/tz/n0": fi(R["tunizi/unique"]), "data/tz/rm": fi(R["tunizi/near_within_removed"] + R["tunizi/tsac_overlap_removed"]),
        "data/na/n0": fi(R["narabizi/unique_latin"]), "data/na/rm": fi(R["narabizi/tsac_overlap_removed"]),
        "data/my/n0": fi(R["myc_lat/in_script"]), "data/my/rm": fi(R["myc_lat/near_within_removed"] + R["myc_lat/tsac_overlap_removed"]),
        "data/myar/n0": fi(R["myc_ar/in_script"]), "data/myar/rm": fi(R["myc_ar/near_within_removed"] + R["myc_ar/tsac_overlap_removed"]),
        "data/human/total": fi(len(TL) + len(TZ) + len(NA) + len(MY)),
        "data/train/mixed": fi(R["tsac_train/mixed"]), "data/train/rm-exact": fi(R["tsac_train/exact_dup_removed"]),
        "data/train/rm-within": fi(R["tsac_train/near_copy_of_val_removed"]),
        "data/tl/mixed": fi(R["tsac_train/mixed"] + R["tsac_test/mixed"]), "data/tl/rm-exact": fi(R["tsac_latin_pool/exact_dup_removed"]),
        "data/tl/rm-within": fi(R["tsac_latin/near_within_removed"]), "data/tl/rm-tsac": "0",
        "data/tl/rm-xscript": fi(R["tsac_latin/cross_script_removed"]),
    }
    for tag, key_ in (("tunizi", "tz"), ("myc_lat", "my"), ("myc_ar", "myar")):
        manual[f"data/{key_}/mixed"] = fi(R[f"{tag}/mixed"]); manual[f"data/{key_}/rm-exact"] = fi(R[f"{tag}/exact_dup_removed"])
        manual[f"data/{key_}/rm-within"] = fi(R[f"{tag}/near_within_removed"]); manual[f"data/{key_}/rm-tsac"] = fi(R[f"{tag}/tsac_overlap_removed"])
        manual[f"data/{key_}/rm-xscript"] = "--"
    manual.update({"data/na/mixed": "0", "data/na/rm-exact": fi(R["narabizi/raw_pairs"] - R["narabizi/unique_latin"]),
                   "data/na/rm-within": "--", "data/na/rm-tsac": fi(R["narabizi/tsac_overlap_removed"]), "data/na/rm-xscript": "--"})
    json.dump(manual, open(os.path.join(OUT, "manual_counts.json"), "w"), indent=1, ensure_ascii=False)
else:
    json.dump({"data/human/total": f"{len(TL) + len(TZ) + len(NA) + len(MY):,}"},
              open(os.path.join(OUT, "manual_counts.json"), "w"), indent=1)
for k, v in report.items():
    print(f"{k}: {v}")

In [ ]:
%%writefile extract.py
"""Cache per-layer pooled sentence vectors (Eq. 1) for every encoder and every set.

For each encoder m and set S writes feats/<m>/<S>.npy with shape [N, L', 2 d_m] in float16
(mean over non-padding tokens || first-token state, for every `--layer_stride`-th layer,
always including the embedding layer 0 and the last layer), and feats/<m>/<S>.frag.npy with
two fragmentation statistics per sentence used by the surface descriptor s(x).
"""
import argparse
import csv
import json
import os
import time

import numpy as np
import torch
from transformers import AutoModel, AutoTokenizer

ENCODERS = {
    "xlmr": "FacebookAI/xlm-roberta-large",
    "me5": "intfloat/multilingual-e5-large",
    "labse": "sentence-transformers/LaBSE",
    # optional members (Section 5.4 / Appendix C)
    "bgem3": "BAAI/bge-m3",
    "marbert": "UBC-NLP/MARBERTv2",
    # cost analysis only (Table 17): timed, never stored or trained on
    "qwen": "Qwen/Qwen2.5-7B",
}
PREFIX = {"me5": "query: "}


def read_texts(path):
    with open(path, encoding="utf-8", newline="") as f:
        return [r["text"] for r in csv.DictReader(f, delimiter="\t")]


@torch.no_grad()
def encode(model, tok, texts, prefix, max_len, batch, device, layers):
    out, frag = [], []
    model.eval()
    wcache = {}

    def n_pieces(w):
        if w not in wcache:
            wcache[w] = len(tok.tokenize(w))
        return wcache[w]
    order = np.argsort([len(t) for t in texts])  # length bucketing for speed
    res = [None] * len(texts); fr = [None] * len(texts)
    for s in range(0, len(texts), batch):
        idx = order[s:s + batch]
        chunk = [prefix + (texts[i] if texts[i].strip() else ".") for i in idx]
        enc = tok(chunk, padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            hs = model(**enc, output_hidden_states=True).hidden_states
        mask = enc["attention_mask"].unsqueeze(-1).float()
        vecs = []
        for l in layers:
            h = hs[l].float()
            mean = (h * mask).sum(1) / mask.sum(1).clamp(min=1)
            vecs.append(torch.cat([mean, h[:, 0]], -1))
        arr = torch.stack(vecs, 1).half().cpu().numpy()
        for j, i in enumerate(idx):
            res[i] = arr[j]
            words = texts[i].split() or ["."]
            pieces = [n_pieces(w) for w in words[:64]]
            fr[i] = (float(np.mean([p > 1 for p in pieces])), float(np.mean(pieces)))
    return np.stack(res), np.array(fr, dtype=np.float32)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data"); ap.add_argument("--out", default="feats")
    ap.add_argument("--encoders", default="xlmr,me5,labse")
    ap.add_argument("--models_dir", default=None, help="local snapshots written by download_models.py")
    ap.add_argument("--layer_stride", type=int, default=2); ap.add_argument("--max_len", type=int, default=64)
    ap.add_argument("--batch", type=int, default=64); ap.add_argument("--sets", default=None)
    ap.add_argument("--cost_only", action="store_true", help="time 1k sentences and batch-1 latency, store nothing")
    args = ap.parse_args()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    sets = args.sets.split(",") if args.sets else sorted(f[:-4] for f in os.listdir(args.data) if f.endswith(".tsv"))
    timing = {}
    for name in args.encoders.split(","):
        src = os.path.join(args.models_dir, name) if args.models_dir else ENCODERS[name]
        tok = AutoTokenizer.from_pretrained(src)
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        kw = {"torch_dtype": torch.float16} if (args.cost_only and device.type == "cuda") else {}
        model = AutoModel.from_pretrained(src, **kw).to(device)
        if args.cost_only:
            texts = read_texts(os.path.join(args.data, "train_ar.tsv"))[:1000]
            L = model.config.num_hidden_layers
            layers = sorted(set(list(range(0, L + 1, args.layer_stride)) + [L]))
            t0 = time.time(); encode(model, tok, texts, "", args.max_len, args.batch, device, layers)
            sec = 1000 * (time.time() - t0) / len(texts)
            t0 = time.time()
            with torch.no_grad():
                for t in texts[:100]:
                    model(**tok([t], return_tensors="pt", truncation=True, max_length=args.max_len).to(device))
            lat = 1000 * (time.time() - t0) / 100
            odir = os.path.join(args.out, name); os.makedirs(odir, exist_ok=True)
            json.dump({"layers": layers, "dim": int(model.config.hidden_size), "sec_per_1k": sec, "latency_ms": lat,
                       "params": int(sum(p.numel() for p in model.parameters())), "cost_only": True},
                      open(os.path.join(odir, "meta.json"), "w"), indent=1)
            print(f"[{name}] cost only: {sec:.1f} s/1k, {lat:.1f} ms/ex"); del model; continue
        L = model.config.num_hidden_layers
        layers = sorted(set(list(range(0, L + 1, args.layer_stride)) + [L]))
        odir = os.path.join(args.out, name); os.makedirs(odir, exist_ok=True)
        n_done, t_done = 0, 0.0
        for s in sets:
            p = os.path.join(odir, f"{s}.npy")
            if os.path.exists(p):
                continue
            texts = read_texts(os.path.join(args.data, f"{s}.tsv"))
            t0 = time.time()
            U, F = encode(model, tok, texts, PREFIX.get(name, ""), args.max_len, args.batch, device, layers)
            t_done += time.time() - t0; n_done += len(texts)
            np.save(p, U); np.save(os.path.join(odir, f"{s}.frag.npy"), F)
            print(f"[{name}] {s}: {U.shape} in {time.time() - t0:.1f}s", flush=True)
        meta = {"layers": layers, "dim": int(model.config.hidden_size), "source": src,
                "params": int(sum(p.numel() for p in model.parameters()))}
        # single-example inference latency (batch size 1), on in-distribution test sentences
        lat_texts = read_texts(os.path.join(args.data, "ar.tsv"))[:100] if os.path.exists(os.path.join(args.data, "ar.tsv")) else []
        if lat_texts:
            with torch.no_grad():
                t0 = time.time()
                for t in lat_texts:
                    enc = tok([PREFIX.get(name, "") + t], truncation=True, max_length=args.max_len, return_tensors="pt").to(device)
                    model(**enc, output_hidden_states=True)
                if device.type == "cuda":
                    torch.cuda.synchronize()
                meta["latency_ms"] = 1000 * (time.time() - t0) / len(lat_texts)
        if n_done:
            meta["sec_per_1k"] = 1000 * t_done / n_done
        old = os.path.join(odir, "meta.json")
        if os.path.exists(old) and "sec_per_1k" in json.load(open(old)) and not n_done:
            meta["sec_per_1k"] = json.load(open(old))["sec_per_1k"]
        json.dump(meta, open(old, "w"), indent=1)
        timing[name] = meta.get("sec_per_1k")
        del model; torch.cuda.empty_cache() if torch.cuda.is_available() else None
    print("extraction s/1k:", timing)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile diafuse_run.py
#!/usr/bin/env python3
"""Train and evaluate DiaFuse-PRC and its baselines on the Tunisian/Arabizi protocol.

Inputs : data/*.tsv (prep_data.py) and feats/<encoder>/<set>.npy (extract.py).
Outputs: preds/<system>/<seed>/<set>.tsv, diag/B12/<seed>/<set>.tsv, emb/<system>/<set>.npy,
         manual.json (counts and costs) -- exactly the inputs of diafuse_eval.py.

    python diafuse_run.py --mode quick            # 3 seeds main, 2 seeds ablations
    python diafuse_run.py --mode full             # 10 / 5 seeds, tuning budget of 20 configs
    python diafuse_run.py --only B6,B12           # subset of systems
Runs whose prediction files already exist are skipped (resumable).
"""
import argparse
import copy
import csv
import itertools
import json
import math
import os
import random
import re
import sys
import time
import unicodedata

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from translit import script_of  # noqa: E402

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
VIEWS = {"AR": "train_ar", "LAT": "train_gs", "INF": "train_inf", "NOISE": "train_noise",
         "CS": "train_cs", "GU": "train_gu"}
FAMILY_OF = {"LAT": "TR", "AR": "TR", "GU": "TR", "INF": "INF", "NOISE": "NOISE", "CS": "CS"}
DEFAULT_POOL = {"xlmr": "mean", "me5": "mean", "labse": "tok", "bgem3": "tok", "marbert": "tok"}
HP = dict(d=512, ds=16, tau_beta0=1.0, tau_beta1=0.5, layer_drop=0.1, gate_hidden=256, tau_g=1.0,
          lam_g=0.01, tau=0.1, lam_cl=0.5, gamma=1.0, rho=0.5, n_pos=10, n_neg=500, pool=256,
          mu=0.9, kappa=50.0, tau_p=0.1, lam_p=0.5, k_hard=20, beta_a=0.5, lam_hm=0.5, n_mix=8,
          eta=1.0, lam_cons=0.5, tau_la=1.0, lam_cal=0.1, kappa_T=100.0, lr=1e-3, wd=5e-4,
          batch=64, epochs=30, p1=3, p2=13)


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


# =========================================================================== data
def read_tsv(path):
    with open(path, encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f, delimiter="\t"))


class Store:
    def __init__(self, data_dir, feats_dir, encoders, layer_stride_check=True):
        self.data_dir, self.feats_dir, self.encoders = data_dir, feats_dir, encoders
        self._rows, self._feat, self._surf = {}, {}, {}
        self.meta = {e: json.load(open(os.path.join(feats_dir, e, "meta.json"))) for e in encoders}

    def rows(self, s):
        if s not in self._rows:
            self._rows[s] = read_tsv(os.path.join(self.data_dir, f"{s}.tsv"))
        return self._rows[s]

    def ids(self, s):
        return [r["id"] for r in self.rows(s)]

    def y(self, s):
        return np.array([int(r["label"]) for r in self.rows(s)])

    def texts(self, s):
        return [r["text"] for r in self.rows(s)]

    def feat(self, s, enc):
        k = (s, enc)
        if k not in self._feat:
            arr = np.load(os.path.join(self.feats_dir, enc, f"{s}.npy"), mmap_mode="r")
            if s.startswith(("train", "val")):  # training/validation views are read at every step: keep in RAM
                arr = np.ascontiguousarray(arr)
            self._feat[k] = arr
        return self._feat[k]

    def has(self, s):
        return os.path.exists(os.path.join(self.data_dir, f"{s}.tsv")) and all(
            os.path.exists(os.path.join(self.feats_dir, e, f"{s}.npy")) for e in self.encoders)

    def pooled(self, s, enc, layer=-1, pool=None):
        U = torch.from_numpy(np.array(self.feat(s, enc)[:, layer], dtype=np.float32))
        d = U.shape[-1] // 2
        pool = pool or DEFAULT_POOL.get(enc, "mean")
        return U[:, :d] if pool == "mean" else U[:, d:]

    def surface(self, s):
        """s(x): 16 cheap surface features (Section 4.3)."""
        if s in self._surf:
            return self._surf[s]
        frags = [np.load(os.path.join(self.feats_dir, e, f"{s}.frag.npy")) for e in self.encoders[:3]]
        while len(frags) < 3:
            frags.append(np.zeros_like(frags[0]))
        out = []
        for i, t in enumerate(self.texts(s)):
            chars = [c for c in t if not c.isspace()] or ["."]
            n = len(chars)
            ar = sum(1 for c in chars if "؀" <= c <= "ۿ") / n
            la = sum(1 for c in chars if c.isascii() and c.isalpha()) / n
            dg = sum(c.isdigit() for c in chars) / n
            toks = t.split() or ["."]
            dl = np.mean([bool(re.search(r"[a-z]", w.lower())) and bool(re.search(r"[235789]", w)) for w in toks])
            acc = np.mean([any(ord(c) > 127 and c.isalpha() and not ("؀" <= c <= "ۿ") for c in w) for w in toks])
            cnt = np.unique(chars, return_counts=True)[1] / n
            ent = float(-(cnt * np.log(cnt)).sum())
            punct = sum(unicodedata.category(c).startswith("P") for c in chars) / n
            rep = np.mean([bool(re.search(r"(.)\1\1", w)) for w in toks])
            f = [ar, la, dg, dl, acc, min(np.mean([len(w) for w in toks]) / 10, 2), min(len(toks) / 50, 1),
                 ent / 5, punct, rep]
            f += [frags[m][i, 0] for m in range(3)] + [min(frags[m][i, 1] / 5, 2) for m in range(3)]
            out.append(f)
        self._surf[s] = torch.tensor(np.array(out, dtype=np.float32))
        return self._surf[s]


class TrainData:
    """Labelled training examples (AR and LAT_syn views) and their consistency partners."""

    def __init__(self, store, syn=True, r_syn=1.0, ir_c=None, twogen=False, families=("TR", "INF", "NOISE", "CS"),
                 seed=0, arabic_only=False):
        rng = np.random.default_rng(1000 + seed)
        y = store.y("train_ar"); n = len(y)
        keep = np.ones(n, bool)
        if ir_c:  # subsample the rarer class so that majority/minority = ir_c
            counts = np.bincount(y, minlength=2); minority = int(np.argmin(counts))
            target = int(counts.max() / ir_c)
            idx = np.flatnonzero(y == minority)
            drop = rng.choice(idx, size=max(0, len(idx) - target), replace=False)
            keep[drop] = False
        src = np.flatnonzero(keep)
        syn_mask = np.zeros(n, bool)
        if syn and not arabic_only:
            k = int(round(r_syn * len(src)))
            syn_mask[rng.choice(src, size=k, replace=False)] = True
        self.items = [("AR", int(i)) for i in src]
        self.items += [("LAT", int(i)) for i in src if syn_mask[i]]
        if twogen:
            self.items += [("GU", int(i)) for i in src if syn_mask[i]]
        self.y_src = y
        self.variety = np.array([0 if v == "AR" else 1 for v, _ in self.items])
        self.label = np.array([y[i] for _, i in self.items])
        self.views_avail = {"AR": np.ones(n, bool), "LAT": syn_mask, "GU": syn_mask if twogen else np.zeros(n, bool),
                            "INF": syn_mask, "NOISE": syn_mask, "CS": syn_mask}
        self.families = set(families)
        self.n_var = 2 if (syn and not arabic_only) else 1
        cells = np.zeros((2, 2))
        for c, v in zip(self.label, self.variety):
            cells[c, v] += 1
        self.cell_counts = cells
        self.prior = np.bincount(self.label, minlength=2) / len(self.label)

    def partner(self, view, i, rng):
        """A transformed view of the same source sentence from an active family (q = 1)."""
        opts = [v for v in VIEWS if v != view and self.views_avail[v][i] and FAMILY_OF[v] in self.families]
        return (opts[rng.integers(len(opts))], i) if opts else None


def stack_layers(store, s, encs, idx=None, layer_mode="both"):
    out = []
    for e in encs:
        U = store.feat(s, e)
        if idx is not None:
            ii = idx.cpu().numpy() if torch.is_tensor(idx) else np.asarray(idx)
            U = U[ii]
        U = torch.from_numpy(np.array(U))
        d = U.shape[-1] // 2
        if layer_mode == "last":
            U = U[:, -1:]
        elif layer_mode == "mean":
            U = U[..., :d]
        elif layer_mode == "tok":
            U = U[..., d:]
        out.append(U)
    return out


# =========================================================================== DiaFuse-PRC
class LayerMix(nn.Module):
    def __init__(self, L, D, d, indepA=False):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(L, D)); self.b = nn.Parameter(torch.zeros(L, d))
        self.omega = nn.Parameter(torch.zeros(L)); self.indepA = indepA
        if indepA:
            self.A = nn.Parameter(torch.randn(L, D, d) / math.sqrt(D))
        else:
            self.A = nn.Linear(D, d, bias=False)

    def forward(self, U, tau_beta, drop=0.0):
        U = F.layer_norm(U, U.shape[-1:]) * self.gamma
        P = (torch.einsum("bld,ldk->blk", U, self.A) if self.indepA else self.A(U)) + self.b
        logits = self.omega / tau_beta
        if self.training and drop > 0 and U.shape[1] > 1:
            mask = torch.rand(U.shape[1], device=U.device) < drop
            if mask.all():
                mask[torch.randint(U.shape[1], (1,))] = False
            logits = logits.masked_fill(mask, -1e9)
        beta = torch.softmax(logits, 0)
        return torch.einsum("l,bld->bd", beta, P), beta


class DiaFuse(nn.Module):
    def __init__(self, shapes, flags, C=2):
        super().__init__()
        d, ds = HP["d"], HP["ds"]
        self.flags = flags; self.M = len(shapes)
        self.mix = nn.ModuleList([LayerMix(L, D, d, flags.get("indepA", False)) for L, D in shapes])
        gin = {"both": self.M * d + ds, "e": self.M * d, "s": ds}[flags.get("gate_input", "both")]
        if flags["fusion"] == "gate" and self.M > 1:
            self.gate = nn.Sequential(nn.Linear(gin, HP["gate_hidden"]), nn.ReLU(), nn.Linear(HP["gate_hidden"], self.M))
        elif flags["fusion"] == "static" and self.M > 1:
            self.static = nn.Linear(self.M * d, d)
        self.W = nn.Linear(d, C)
        self.proj_f = nn.Sequential(nn.Linear(d, 256), nn.ReLU(), nn.Linear(256, 128))
        self.proj_h = nn.Sequential(nn.Linear(d, 256), nn.ReLU(), nn.Linear(256, 128))

    def forward(self, Us, s, tau_beta=0.5):
        es = [m(U.to(DEV).float(), tau_beta, HP["layer_drop"])[0] for m, U in zip(self.mix, Us)]
        B = es[0].shape[0]
        if self.M == 1:
            return es[0], torch.ones(B, 1, device=DEV), es
        if self.flags["fusion"] == "static":
            return self.static(torch.cat(es, -1)), torch.full((B, self.M), 1.0 / self.M, device=DEV), es
        gi = self.flags.get("gate_input", "both")
        parts = []
        if gi in ("both", "e"):
            parts += [F.layer_norm(e, e.shape[-1:]) for e in es]
        if gi in ("both", "s"):
            parts.append(s.to(DEV))
        alpha = torch.softmax(self.gate(torch.cat(parts, -1)) / HP["tau_g"], -1)
        z = torch.einsum("bm,bmd->bd", alpha, torch.stack(es, 1))
        return z, alpha, es

    def prototypes(self):
        return F.normalize(self.proj_h(self.W.weight), dim=-1)


def js_div(p, q):
    m = 0.5 * (p + q)
    kl = lambda a, b: (a * (torch.log(a + 1e-8) - torch.log(b + 1e-8))).sum(-1)
    return 0.5 * kl(p, m) + 0.5 * kl(q, m)


def mmce(conf, correct, width=0.4):
    k = torch.exp(-torch.abs(conf[:, None] - conf[None, :]) / width)
    r = (correct - conf)
    return (r[:, None] * r[None, :] * k).mean()


class DiaFuseTrainer:
    """One training run of DiaFuse-PRC (or of a SharpReCL-style variant, through flags)."""

    def __init__(self, store, td, flags, seed, encs, input_mode="layers"):
        self.store, self.td, self.flags, self.seed, self.encs = store, td, flags, seed, encs
        self.input_mode = input_mode  # "layers" or "pooled" (B10: concatenated default vectors)
        set_seed(seed)
        shapes = [(U.shape[1], U.shape[2]) for U in self._inputs("train_ar", [0])]
        self.model = DiaFuse(shapes, flags).to(DEV)
        self.log_prior = torch.log(torch.tensor(td.prior, dtype=torch.float32, device=DEV).clamp(min=1e-6))
        V = td.n_var
        self.delta = torch.zeros(2, 2, 128, device=DEV)
        cnt = torch.tensor(td.cell_counts, dtype=torch.float32, device=DEV)
        kappa = flags.get("kappa", HP["kappa"])
        self.r_cv = cnt / (cnt + kappa) if math.isfinite(kappa) else torch.zeros_like(cnt)
        self.cell_prior = cnt / cnt.sum()

    # ---------------------------------------------------------------- inputs
    def _inputs(self, s, idx=None):
        if self.input_mode == "pooled":
            X = torch.cat([self.store.pooled(s, e) for e in self.encs], -1)
            X = X if idx is None else X[idx]
            return [X.unsqueeze(1)]
        return stack_layers(self.store, s, self.encs, idx, self.flags.get("layer_mode", "both"))

    def _surf(self, s, idx):
        S = self.store.surface(s)
        return S[idx] if idx is not None else S

    def forward_items(self, items, tau_beta, train=True):
        """items: list of (view, row). Groups by view to gather features."""
        by = {}
        for k, (v, i) in enumerate(items):
            by.setdefault(v, []).append((k, i))
        zs, als = [None] * len(items), [None] * len(items)
        self.model.train(train)
        for v, lst in by.items():
            pos = [k for k, _ in lst]; rows = torch.tensor([i for _, i in lst])
            z, a, _ = self.model(self._inputs(VIEWS[v], rows), self._surf(VIEWS[v], rows), tau_beta)
            for j, k in enumerate(pos):
                zs[k] = z[j]; als[k] = a[j]
        return torch.stack(zs), torch.stack(als)

    def encode_set(self, s, tau_beta, bs=512):
        self.model.eval(); n = len(self.store.ids(s)); Z, A = [], []
        with torch.no_grad():
            for i in range(0, n, bs):
                idx = torch.arange(i, min(n, i + bs))
                z, a, _ = self.model(self._inputs(s, idx), self._surf(s, idx), tau_beta)
                Z.append(z); A.append(a)
        return torch.cat(Z), torch.cat(A)

    # ---------------------------------------------------------------- losses
    def ce_la(self, logits, y):
        return F.cross_entropy(logits + HP["tau_la"] * self.log_prior, y)

    def proto_cv(self):
        p = self.model.prototypes()  # [C,128]
        mode = self.flags.get("proto", "hier")
        if mode == "class" or self.td.n_var == 1:
            return p[:, None, :].expand(2, 2, 128)
        pcv = p[:, None, :] + self.r_cv[..., None] * self.delta
        return F.normalize(pcv, dim=-1)

    def cl_loss(self, zh, y, v, bank_z, bank_y, bank_v, syn, r_t):
        """Cell-rebalanced contrastive loss (Eq. 11) with targets from the epoch bank."""
        tau, gamma, rho = self.flags.get("tau", HP["tau"]), HP["gamma"], HP["rho"]
        classes_only = self.flags.get("cells", "cv") == "class"
        V = 1 if classes_only else self.td.n_var
        bv = bank_v if not classes_only else torch.zeros_like(bank_v)
        pools, pool_cell = [], []
        for c in range(2):
            for vv in range(V):
                ids = torch.nonzero((bank_y == c) & (bv == vv)).flatten()
                if len(ids) == 0:
                    continue
                pools.append(ids[torch.randint(len(ids), (HP["pool"],), device=ids.device)]); pool_cell.append((c, vv))
        T_idx = torch.cat(pools)
        T_c = torch.tensor([c for c, _ in pool_cell], device=DEV).repeat_interleave(HP["pool"])
        T_v = torch.tensor([vv for _, vv in pool_cell], device=DEV).repeat_interleave(HP["pool"])
        T_z = F.normalize(self.model.proj_f(bank_z[T_idx]), dim=-1)
        # anchors: batch + prototypes of every cell
        anchors, ay, av, aw = [zh], [y], [v if not classes_only else torch.zeros_like(v)], []
        cp = self.cell_prior if not classes_only else self.cell_prior.sum(1, keepdim=True)
        w = cp[y, (v if not classes_only else torch.zeros_like(v))].clamp(min=1e-6) ** (-rho)
        aw.append(w / w.mean())
        if self.flags.get("proto", "hier") != "none":
            P = self.proto_cv()
            for c in range(2):
                for vv in range(V):
                    anchors.append(P[c, vv][None]); ay.append(torch.tensor([c], device=DEV))
                    av.append(torch.tensor([vv], device=DEV)); aw.append(torch.ones(1, device=DEV))
        A = torch.cat(anchors); Ay = torch.cat(ay); Av = torch.cat(av); Aw = torch.cat(aw)
        S = A @ T_z.T / tau  # [nA, nT]
        nA = A.shape[0]
        # positives: n_pos per anchor from same-class pools, variety uniform (zeta = 0)
        same = (T_c[None, :] == Ay[:, None]).float()
        other = 1 - same
        pos_idx = torch.multinomial(same + 1e-12 * other, HP["n_pos"], replacement=True)
        neg_idx = torch.multinomial(other + 1e-12 * same, min(HP["n_neg"], int(other.sum(1).min().item())) or 1,
                                    replacement=True)
        s_pos = S.gather(1, pos_idx); s_neg = S.gather(1, neg_idx)
        w_pos = 1 + gamma * (T_v[pos_idx] != Av[:, None]).float()
        extra_pos, extra_neg, extra_wp = [], [], []
        if syn is not None and r_t > 0:
            n_sp = max(1, round(r_t * 5)); n_sn = max(1, round(r_t * 50))
            sp = torch.zeros(nA, n_sp, device=DEV); sn = torch.zeros(nA, n_sn, device=DEV); wp = torch.ones(nA, n_sp, device=DEV)
            for c in range(2):
                m = Ay == c
                if not m.any():
                    continue
                Pz, Pcross, Nz = syn[c]
                if Pz is not None:
                    j = torch.randint(len(Pz), (n_sp,), device=DEV)
                    sp[m] = A[m] @ Pz[j].T / tau; wp[m] = (1 + gamma * Pcross[j].float())[None, :].expand(int(m.sum()), -1)
                if Nz is not None:
                    j = torch.randint(len(Nz), (n_sn,), device=DEV)
                    sn[m] = A[m] @ Nz[j].T / tau
            s_pos = torch.cat([s_pos, sp], 1); s_neg = torch.cat([s_neg, sn], 1); w_pos = torch.cat([w_pos, wp], 1)
        denom = torch.logsumexp(torch.cat([s_pos, s_neg], 1), 1, keepdim=True)
        per = (w_pos * (denom - s_pos)).sum(1) / w_pos.sum(1)
        return (Aw * per).mean()

    def hard_sets(self, bank_z, bank_y, bank_v, noisy):
        """Algorithm 3: hard positives/negatives per class from the epoch bank."""
        with torch.no_grad():
            zh = F.normalize(self.model.proj_f(bank_z), dim=-1)
            logits = self.model.W(bank_z); p = torch.softmax(logits, -1)
            P = self.proto_cv(); pc = self.model.prototypes()
            vv = bank_v.clamp(max=P.shape[1] - 1)
            dist = 1 - (zh * P[bank_y, vv]).sum(-1)
            ent = -(p * torch.log(p + 1e-8)).sum(-1)
            fy = logits.gather(1, bank_y[:, None]).squeeze(1)
            margin = fy - logits.masked_fill(F.one_hot(bank_y, 2).bool(), -1e9).max(1).values
            zs = lambda x: (x - x.mean()) / (x.std() + 1e-8)
            mode = self.flags.get("hardness", "all")
            h = {"all": zs(dist) + zs(ent) + zs(-margin), "dist": zs(dist), "ent": zs(ent), "mar": zs(-margin),
                 "proto": zs(1 - (zh * pc[bank_y]).sum(-1))}[mode]
            h = h.masked_fill(noisy, -1e9)
            k = HP["k_hard"]; out = {}
            for c in range(2):
                posc = torch.nonzero(bank_y == c).flatten()
                negc = torch.nonzero(bank_y != c).flatten()
                hp = posc[torch.topk(h[posc], min(k, len(posc))).indices]
                conf = (zh[negc] @ pc[c]) + p[negc, c]
                hn = negc[torch.topk(conf.masked_fill(noisy[negc], -1e9), min(k, len(negc))).indices]
                out[c] = (hp, hn)
        return out

    def make_mixes(self, hs, bank_z, bank_y, bank_v):
        beta = torch.distributions.Beta(HP["beta_a"], HP["beta_a"])
        syn, boundary = {}, []
        for c in range(2):
            hp, hn = hs[c]; n = HP["n_mix"]
            Pz = Pcross = Nz = None
            if len(hp) >= 2:
                i = hp[torch.randint(len(hp), (n,), device=DEV)]
                # prefer partners of a different variety
                j = torch.empty_like(i)
                for t in range(n):
                    cand = hp[bank_v[hp] != bank_v[i[t]]]
                    cand = cand if len(cand) else hp
                    j[t] = cand[torch.randint(len(cand), (1,), device=DEV)]
                lam = beta.sample((n, 1)).to(DEV)
                Pz = F.normalize(self.model.proj_f(lam * bank_z[i] + (1 - lam) * bank_z[j]), dim=-1)
                Pcross = bank_v[i] != bank_v[j]
            if len(hn) >= 2:
                i = hn[torch.randint(len(hn), (n,), device=DEV)]; j = hn[torch.randint(len(hn), (n,), device=DEV)]
                lam = beta.sample((n, 1)).to(DEV)
                Nz = F.normalize(self.model.proj_f(lam * bank_z[i] + (1 - lam) * bank_z[j]), dim=-1)
            syn[c] = (Pz, Pcross, Nz)
            if len(hp) and len(hn):
                i = hp[torch.randint(len(hp), (2 * n,), device=DEV)]; j = hn[torch.randint(len(hn), (2 * n,), device=DEV)]
                lam = beta.sample((2 * n, 1)).to(DEV)
                zt = lam * bank_z[i] + (1 - lam) * bank_z[j]
                yt = lam * F.one_hot(torch.full((2 * n,), c, device=DEV), 2).float() + (1 - lam) * F.one_hot(bank_y[j], 2).float()
                boundary.append((zt, yt))
        return syn, boundary

    # ---------------------------------------------------------------- training
    def dev_score(self, tau_beta):
        scores = []
        for s in self.dev_sets:
            z, _ = self.encode_set(s, tau_beta)
            pred = self.model.W(z).argmax(-1).cpu().numpy()
            scores.append(macro_f1(self.store.y(s), pred))
        return float(np.mean(scores))

    def fit(self, dev_sets):
        self.dev_sets = dev_sets
        td, fl, E = self.td, self.flags, HP["epochs"]
        rng = np.random.default_rng(self.seed)
        opt = torch.optim.AdamW(self.model.parameters(), lr=HP["lr"], weight_decay=HP["wd"])
        n = len(td.items); y_all = torch.tensor(td.label, device=DEV); v_all = torch.tensor(td.variety, device=DEV)
        flagged = np.zeros(n); best, best_state = -1, None
        lam_cl = fl.get("lam_cl", HP["lam_cl"]); lam_cons = fl.get("lam_cons", HP["lam_cons"])
        for ep in range(E):
            tau_beta = HP["tau_beta0"] - (HP["tau_beta0"] - HP["tau_beta1"]) * ep / max(1, E - 1)
            phase = 1 if ep < HP["p1"] else (2 if ep < HP["p2"] else 3)
            cal_on = fl.get("cal", True) and phase == 3
            perm = rng.permutation(n)
            cal_slice = perm[: n // 10] if cal_on else np.array([], int)
            main = perm[n // 10:] if cal_on else perm
            bank_z = None; syn = None; boundary = []
            if phase >= 2:
                with torch.no_grad():
                    Z = []
                    for i in range(0, n, 512):
                        z, _ = self.forward_items(td.items[i:i + 512], tau_beta, train=False); Z.append(z)
                    bank_z = torch.cat(Z)
                    loss_i = F.cross_entropy(self.model.W(bank_z) + HP["tau_la"] * self.log_prior, y_all, reduction="none")
                    top = loss_i >= torch.quantile(loss_i, 0.98)
                    flagged += top.cpu().numpy()
                noisy = torch.tensor(flagged > (ep + 1) / 2, device=DEV)
                if phase == 3 and fl.get("hm", True):
                    hs = self.hard_sets(bank_z, y_all, v_all, noisy)
            r_t = 0.5 + 0.5 * (ep - HP["p2"]) / max(1, E - HP["p2"]) if phase == 3 else 0.0
            ramp = min(1.0, (ep - HP["p1"] + 1) / (HP["p2"] - HP["p1"])) if phase >= 2 else 0.0
            for b in range(0, len(main), HP["batch"]):
                bi = main[b:b + HP["batch"]]
                items = [td.items[i] for i in bi]
                y = y_all[bi]; v = v_all[bi]
                z, alpha = self.forward_items(items, tau_beta)
                logits = self.model.W(z)
                loss = self.ce_la(logits, y)
                if self.model.M > 1 and fl["fusion"] == "gate":
                    abar = alpha.mean(0)
                    loss = loss + HP["lam_g"] * (abar * torch.log(abar * self.model.M + 1e-8)).sum()
                if phase >= 2:
                    zh = F.normalize(self.model.proj_f(z), dim=-1)
                    with torch.no_grad():  # EMA variety offsets (Eq. 9)
                        pc = self.model.prototypes()
                        for c in range(2):
                            for vv in range(td.n_var):
                                m = (y == c) & (v == vv)
                                if m.any():
                                    self.delta[c, vv] = HP["mu"] * self.delta[c, vv] + (1 - HP["mu"]) * (zh[m] - pc[c]).mean(0)
                    if fl.get("proto", "hier") != "none":
                        P = self.proto_cv()
                        vv = v.clamp(max=P.shape[1] - 1)
                        logits_p = torch.einsum("bk,bck->bc", zh, P[:, vv].permute(1, 0, 2)) / HP["tau_p"]
                        loss = loss + HP["lam_p"] * F.cross_entropy(logits_p, y)
                    if phase == 3 and fl.get("hm", True):
                        syn, boundary = self.make_mixes(hs, bank_z, y_all, v_all)
                    if lam_cl > 0:
                        loss = loss + lam_cl * self.cl_loss(zh, y, v, bank_z, y_all, v_all, syn if phase == 3 else None, r_t)
                    if fl.get("cons", True):
                        pairs = [(k, td.partner(vw, i, rng)) for k, (vw, i) in enumerate(items)]
                        pairs = [(k, p) for k, p in pairs if p is not None]
                        if pairs:
                            ks = torch.tensor([k for k, _ in pairs], device=DEV)
                            z2, a2 = self.forward_items([p for _, p in pairs], tau_beta)
                            terms = fl.get("cons_terms", "both")
                            lc = 0
                            if terms in ("both", "emb"):
                                lc = lc + (1 - F.cosine_similarity(z[ks], z2, dim=-1))
                            if terms in ("both", "pred"):
                                lc = lc + HP["eta"] * js_div(torch.softmax(logits[ks], -1), torch.softmax(self.model.W(z2), -1))
                            loss = loss + lam_cons * ramp * lc.mean()
                            if fl.get("gatecons", False) and self.model.M > 1:
                                loss = loss + 0.1 * ((alpha[ks] - a2) ** 2).sum(-1).mean()
                if phase >= 2 and fl.get("xling", False) and self.store.has("train_madar_tn"):
                    npairs = len(self.store.ids("train_madar_tn"))
                    pi = torch.tensor(rng.choice(npairs, min(HP["batch"], npairs), replace=False))
                    za, _, _ = self.model(self._inputs("train_madar_tn", pi), self._surf("train_madar_tn", pi), tau_beta)
                    zb, _, _ = self.model(self._inputs("train_madar_msa", pi), self._surf("train_madar_msa", pi), tau_beta)
                    lg = F.normalize(self.model.proj_f(za), dim=-1) @ F.normalize(self.model.proj_f(zb), dim=-1).T / 0.05
                    tgt = torch.arange(len(pi), device=DEV)
                    loss = loss + fl.get("lam_x", 0.1) * 0.5 * (F.cross_entropy(lg, tgt) + F.cross_entropy(lg.T, tgt))
                if phase == 3 and fl.get("hm", True) and boundary:
                    zt = torch.cat([b_[0] for b_ in boundary]); yt = torch.cat([b_[1] for b_ in boundary])
                    loss = loss + HP["lam_hm"] * (-(yt * F.log_softmax(self.model.W(zt), -1)).sum(-1).mean())
                if cal_on and len(cal_slice):
                    ci = cal_slice[rng.choice(len(cal_slice), min(HP["batch"], len(cal_slice)), replace=False)]
                    zc, _ = self.forward_items([td.items[i] for i in ci], tau_beta)
                    lg = self.model.W(zc); pr = torch.softmax(lg + HP["tau_la"] * self.log_prior, -1)
                    pred = lg.argmax(-1); conf = pr.gather(1, pred[:, None]).squeeze(1)
                    loss = loss + HP["lam_cal"] * mmce(conf, (pred == y_all[ci]).float())
                opt.zero_grad(); loss.backward(); opt.step()
            if ep >= HP["p1"] - 1:
                sc = self.dev_score(tau_beta)
                if sc > best:
                    best, best_state = sc, (copy.deepcopy(self.model.state_dict()), self.delta.clone(), tau_beta)
        self.model.load_state_dict(best_state[0]); self.delta = best_state[1]; self.tau_beta = best_state[2]
        self.dev = best
        return self

    # ---------------------------------------------------------------- calibration + prediction
    def calibrate(self, prior_corr=True):
        logits, vlab, feats = [], [], []
        for k, s in enumerate(self.dev_sets):
            z, a = self.encode_set(s, self.tau_beta)
            f = self.model.W(z)
            logits.append(f.detach()); vlab += [k] * len(f)
            feats.append(torch.cat([a, self.store.surface(s).to(DEV)], -1).cpu().numpy())
        ys = np.concatenate([self.store.y(s) for s in self.dev_sets])
        L = torch.cat(logits); vlab = np.array(vlab)
        self.cal = {}
        for pc in (True, False):
            G = L + HP["tau_la"] * self.log_prior if pc else L
            T = fit_temperature(G, ys)
            Tv = {}
            for k in range(len(self.dev_sets)):
                m = vlab == k
                Tk = fit_temperature(G[m], ys[m]); nk = int(m.sum())
                Tv[k] = math.exp((nk * math.log(Tk) + HP["kappa_T"] * math.log(T)) / (nk + HP["kappa_T"]))
            self.cal[pc] = (T, Tv)
        X = np.concatenate(feats)
        self.probe = LogisticRegression(max_iter=2000).fit(X, vlab) if len(self.dev_sets) > 1 else None
        # off-support threshold on validation (both views)
        P = self.proto_cv().reshape(-1, 128)
        sims = []
        for s in self.dev_sets:
            z, _ = self.encode_set(s, self.tau_beta)
            sims.append((F.normalize(self.model.proj_f(z), dim=-1) @ P.T).max(1).values.detach().cpu().numpy())
        self.val_maxproto = np.concatenate(sims)
        return self

    def predict(self, s, variety_wise=True):
        with torch.no_grad():
            z, a = self.encode_set(s, self.tau_beta)
            f = self.model.W(z)
            feats = torch.cat([a, self.store.surface(s).to(DEV)], -1).cpu().numpy()
            if self.probe is not None:
                pv = self.probe.predict_proba(feats); vhat = pv.argmax(1); plat = pv[:, -1]
            else:
                vhat = np.zeros(len(f), int); plat = np.zeros(len(f))
            out = {"pred": f.argmax(-1).cpu().numpy(), "plat": plat, "alpha": a.cpu().numpy(), "z": z.cpu().numpy()}
            P = self.proto_cv().reshape(-1, 128)
            out["maxproto"] = (F.normalize(self.model.proj_f(z), dim=-1) @ P.T).max(1).values.cpu().numpy()
            for pc, tag in ((True, ""), (False, "_nopc")):
                G = f + HP["tau_la"] * self.log_prior if pc else f
                T, Tv = self.cal[pc]
                out["p_raw" + tag] = torch.softmax(G, -1)[:, 1].cpu().numpy()
                out["p_glob" + tag] = torch.softmax(G / T, -1)[:, 1].cpu().numpy()
                tv = torch.tensor([Tv.get(int(k), T) for k in vhat], device=DEV, dtype=G.dtype)[:, None]
                out["p_var" + tag] = (torch.softmax(G / tv, -1)[:, 1].cpu().numpy() if variety_wise
                                      else out["p_glob" + tag])
        return out


def fit_temperature(logits, y):
    logits = logits.detach().float(); y = torch.tensor(y, device=logits.device)
    best, bestT = 1e9, 1.0
    for T in np.exp(np.linspace(np.log(0.05), np.log(20), 120)):
        nll = F.cross_entropy(logits / float(T), y).item()
        if nll < best:
            best, bestT = nll, float(T)
    return bestT


def macro_f1(y, p):
    f = []
    for c in (0, 1):
        tp = np.sum((p == c) & (y == c)); fp = np.sum((p == c) & (y != c)); fn = np.sum((p != c) & (y == c))
        f.append(0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn))
    return float(np.mean(f))


# =========================================================================== baselines on pooled vectors
class MLPHead(nn.Module):
    def __init__(self, din, hidden=256, dropout=0.1, C=2, proj=False):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(din, hidden), nn.ReLU(), nn.Dropout(dropout))
        self.W = nn.Linear(hidden, C)
        self.proj = nn.Sequential(nn.Linear(hidden, 128), nn.ReLU(), nn.Linear(128, 128)) if proj else None

    def forward(self, x):
        h = self.body(x)
        return self.W(h), h


class Fuse7(nn.Module):
    def __init__(self, dims, mode, hidden=256, dropout=0.1):
        super().__init__()
        self.mode = mode; self.dims = dims
        self.P = nn.ModuleList([nn.Linear(d, 512) for d in dims]); self.head = MLPHead(512, hidden, dropout)

    def forward(self, x):
        parts = torch.split(x, self.dims, -1); es = [p(t) for p, t in zip(self.P, parts)]
        s = sum(es); h = es[0]
        for e in es[1:]:
            h = h * e
        z = {"sum": s, "hadamard": h, "residual": s + h}[self.mode]
        return self.head(z)


def supcon(zh, y, tau=0.1):
    S = zh @ zh.T / tau
    eye = torch.eye(len(y), device=DEV, dtype=torch.bool)
    S = S.masked_fill(eye, -1e9)
    pos = (y[:, None] == y[None, :]) & ~eye
    logp = S - torch.logsumexp(S, 1, keepdim=True)
    cnt = pos.sum(1).clamp(min=1)
    return -((logp * pos).sum(1) / cnt).mean()


def bcl_loss(zh, y, protos, tau=0.1):
    """Balanced contrastive learning: class-averaged denominators, class prototypes as anchors' targets."""
    allz = torch.cat([zh, protos]); ally = torch.cat([y, torch.arange(2, device=DEV)])
    S = zh @ allz.T / tau
    n = len(y); eye = torch.zeros_like(S, dtype=torch.bool); eye[:, :n] = torch.eye(n, device=DEV, dtype=torch.bool)
    expS = torch.exp(S - S.max(1, keepdim=True).values).masked_fill(eye, 0)
    denom = 0
    for c in range(2):
        mc = (ally == c)[None, :].expand_as(S) & ~eye
        denom = denom + (expS * mc).sum(1) / mc.sum(1).clamp(min=1)
    pos = (ally[None, :] == y[:, None]) & ~eye
    logp = torch.log(expS + 1e-12) - torch.log(denom[:, None] + 1e-12)
    return -((logp * pos).sum(1) / pos.sum(1).clamp(min=1)).mean()


def train_pooled(Xtr, ytr, dev_list, seed, kind="mlp", hp=None, extra=None, prior_la=False):
    """Generic trainer for B0/B2-B9/CEIL on fixed vectors. dev_list: [(X, y)]."""
    hp = {"lr": 1e-3, "dropout": 0.1, "hidden": 256, "wd": 5e-4, **(hp or {})}
    set_seed(seed)
    din = Xtr.shape[1]
    if kind.startswith("fuse7:"):
        model = Fuse7(extra["dims"], kind.split(":")[1], hp["hidden"], hp["dropout"]).to(DEV)
    else:
        model = MLPHead(din, hp["hidden"], hp["dropout"], proj=kind in ("supcon", "bcl")).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=hp["lr"], weight_decay=hp["wd"])
    Xtr = Xtr.to(DEV); ytr_t = torch.tensor(ytr, device=DEV)
    prior = torch.log(torch.tensor(np.bincount(ytr, minlength=2) / len(ytr), device=DEV, dtype=torch.float32))
    best, state = -1, None; rng = np.random.default_rng(seed)
    for ep in range(HP["epochs"]):
        model.train()
        perm = rng.permutation(len(ytr))
        for b in range(0, len(perm), HP["batch"]):
            bi = torch.tensor(perm[b:b + HP["batch"]], device=DEV)
            logits, h = model(Xtr[bi]); yb = ytr_t[bi]
            loss = F.cross_entropy(logits + (prior if prior_la else 0), yb)
            if kind == "supcon":
                loss = loss + 0.5 * supcon(F.normalize(model.proj(h), dim=-1), yb)
            if kind == "bcl":
                protos = F.normalize(model.proj(model.W.weight), dim=-1)
                loss = loss + 0.5 * bcl_loss(F.normalize(model.proj(h), dim=-1), yb, protos)
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            sc = np.mean([macro_f1(y, model(X.to(DEV))[0].argmax(-1).cpu().numpy()) for X, y in dev_list])
        if sc > best:
            best, state = sc, copy.deepcopy(model.state_dict())
    model.load_state_dict(state); model.eval()
    model.dev = best; model.prior = prior if prior_la else torch.zeros(2, device=DEV)
    return model


def pooled_predict(model, X, T=1.0, Tv=None, varieties=None):
    with torch.no_grad():
        f = model(X.to(DEV))[0]
        G = f + model.prior
        out = {"pred": f.argmax(-1).cpu().numpy(), "p_raw": torch.softmax(G, -1)[:, 1].cpu().numpy(),
               "p_glob": torch.softmax(G / T, -1)[:, 1].cpu().numpy()}
        if Tv is not None and varieties is not None:
            tv = torch.tensor([Tv[v] for v in varieties], device=DEV, dtype=G.dtype)[:, None]
            out["p_var"] = torch.softmax(G / tv, -1)[:, 1].cpu().numpy()
        else:
            out["p_var"] = out["p_glob"]
        out["logits"] = f.cpu().numpy()
    return out


# =========================================================================== orchestration
class Runner:
    def __init__(self, args):
        self.a = args
        self.encs = args.encoders.split(",")
        self.store = Store(args.data, args.feats, self.encs)
        self.eval_sets = [s for s in sorted(f[:-4] for f in os.listdir(args.data) if f.endswith(".tsv"))
                          if not s.startswith(("train", "val", "tt-")) and self.store.has(s)]
        self.costs = {}
        self.devfile = os.path.join(args.out, "dev_scores.json")
        self.devlog = json.load(open(self.devfile)) if os.path.exists(self.devfile) else {}

    def log_dev(self, system, seed, score):
        self.devlog[f"{system}/{seed}"] = score
        os.makedirs(self.a.out, exist_ok=True); json.dump(self.devlog, open(self.devfile, "w"), indent=0)

    def dev_of(self, system):
        v = [x for k, x in self.devlog.items() if k.rsplit("/", 1)[0] == system]
        return float(np.mean(v)) if v else None

    def out(self, system, seed, s):
        return os.path.join(self.a.out, "preds", system, str(seed), f"{s}.tsv")

    def done(self, system, seed):
        return os.path.exists(self.out(system, seed, "my"))

    def write_preds(self, system, seed, s, res, ids=None):
        p = self.out(system, seed, s); os.makedirs(os.path.dirname(p), exist_ok=True)
        ids = ids or self.store.ids(s)
        with open(p, "w", encoding="utf-8") as f:
            f.write("id\tpred\tp_raw\tp_glob\tp_var\n")
            for k, i in enumerate(ids):
                f.write(f"{i}\t{int(res['pred'][k])}\t{res['p_raw'][k]:.6f}\t{res['p_glob'][k]:.6f}\t{res['p_var'][k]:.6f}\n")

    def varieties(self, s):
        return [1 if script_of(t) == "lat" else 0 for t in self.store.texts(s)]

    # ------------------------------------------------------------ pooled-vector systems
    def vecs(self, s, spec):
        """spec: list of (enc, layer, pool)."""
        return torch.cat([self.store.pooled(s, e, l, p) for e, l, p in spec], -1)

    def run_pooled(self, system, spec, seeds, kind="mlp", hp=None, train_views=("train_ar", "train_gs"),
                   dev_sets=("val_ar", "val_gs"), prior_la=False, extra=None, eval_map=None, tune=False):
        if tune and self.a.mode == "full":
            hp = self.tune_pooled(spec, kind, train_views, dev_sets, prior_la, extra)
        Xtr = torch.cat([self.vecs(v, spec) for v in train_views]); ytr = np.concatenate([self.store.y(v) for v in train_views])
        devs = [(self.vecs(s, spec), self.store.y(s)) for s in dev_sets]
        devs_score = []
        for seed in seeds:
            if self.done(system, seed):
                continue
            t0 = time.time()
            m = train_pooled(Xtr, ytr, devs, seed, kind, hp, extra, prior_la)
            self.costs.setdefault(system, []).append((time.time() - t0) / 60)
            devs_score.append(m.dev); self.log_dev(system, seed, m.dev)
            G = torch.cat([m(X.to(DEV))[0].detach() + m.prior for X, _ in devs])
            yd = np.concatenate([y for _, y in devs]); T = fit_temperature(G, yd)
            Tv = {}
            for k, (X, y) in enumerate(devs):
                Gk = m(X.to(DEV))[0].detach() + m.prior; Tk = fit_temperature(Gk, y)
                Tv[k] = math.exp((len(y) * math.log(Tk) + HP["kappa_T"] * math.log(T)) / (len(y) + HP["kappa_T"]))
            vmap = {0: Tv.get(0, T), 1: Tv.get(1, Tv.get(0, T))}
            for s in self.eval_sets:
                src = (eval_map or {}).get(s, s)
                if not self.store.has(src):
                    continue
                res = pooled_predict(m, self.vecs(src, spec), T, vmap, self.varieties(src))
                self.write_preds(system, seed, s, res, self.store.ids(s))
            if system == "B6" and seed == seeds[0]:
                self.save_emb("B6", spec)
        return self.dev_of(system)

    def tune_pooled(self, spec, kind, train_views, dev_sets, prior_la, extra):
        grid = list(itertools.product([1e-3, 3e-4], [0.1, 0.3], [256, 512], [5e-4, 1e-2]))[:20]
        Xtr = torch.cat([self.vecs(v, spec) for v in train_views]); ytr = np.concatenate([self.store.y(v) for v in train_views])
        devs = [(self.vecs(s, spec), self.store.y(s)) for s in dev_sets]
        best, best_hp = -1, None
        for lr, do, hid, wd in grid:
            hp = {"lr": lr, "dropout": do, "hidden": hid, "wd": wd}
            sc = train_pooled(Xtr, ytr, devs, 0, kind, hp, extra, prior_la).dev
            if sc > best:
                best, best_hp = sc, hp
        return best_hp

    def save_emb(self, system, spec=None, trainer=None):
        d = os.path.join(self.a.out, "emb", system); os.makedirs(d, exist_ok=True)
        for s in ["train_gs", "ar", "gs", "gu", "tl", "tz", "na", "my", "myar"]:
            if not self.store.has(s):
                continue
            if trainer is not None:
                z, _ = trainer.encode_set(s, trainer.tau_beta); np.save(os.path.join(d, f"{s}.npy"), z.cpu().numpy())
            else:
                np.save(os.path.join(d, f"{s}.npy"), self.vecs(s, spec).numpy())

    def default_spec(self, encs=None):
        return [(e, -1, DEFAULT_POOL.get(e, "mean")) for e in (encs or self.encs[:3])]

    # ------------------------------------------------------------ DiaFuse family
    def run_diafuse(self, system, seeds, flags, data_kw=None, encs=None, input_mode="layers", diag=False,
                    variety_wise=True, also_nopc=False, dev_sets=None):
        flags = {"fusion": "gate", **flags}
        encs = encs or self.encs[:3]
        devs = []
        for seed in seeds:
            if self.done(system, seed):
                continue
            td = TrainData(self.store, seed=seed, **(data_kw or {}))
            ds = dev_sets or (["val_ar", "val_gs"] if td.n_var > 1 else ["val_ar"])
            t0 = time.time()
            tr = DiaFuseTrainer(self.store, td, flags, seed, encs, input_mode).fit(ds).calibrate()
            self.costs.setdefault(system, []).append((time.time() - t0) / 60)
            devs.append(tr.dev); self.log_dev(system, seed, tr.dev)
            self.costs[f"{system}/params"] = sum(p.numel() for p in tr.model.parameters())
            for s in self.eval_sets:
                res = tr.predict(s, variety_wise=variety_wise and td.n_var > 1)
                self.write_preds(system, seed, s, res)
                if also_nopc:
                    self.write_preds(system + "-nopc", seed, s, {"pred": res["pred"], "p_raw": res["p_raw_nopc"],
                                                                 "p_glob": res["p_glob_nopc"], "p_var": res["p_var_nopc"]})
                if diag:
                    self.write_diag(system, seed, s, res)
            if diag:
                vres = {k: np.concatenate([tr.predict(s)[k] for s in ds]) for k in ("alpha", "plat", "maxproto")}
                ids = sum([self.store.ids(s) for s in ds], [])
                self.write_diag(system, seed, "val", vres, ids)
                if seed == seeds[0]:
                    self.save_emb(system, trainer=tr)
                    self.head_latency(tr)
        return self.dev_of(system)

    def write_diag(self, system, seed, s, res, ids=None):
        p = os.path.join(self.a.out, "diag", system, str(seed), f"{s}.tsv"); os.makedirs(os.path.dirname(p), exist_ok=True)
        ids = ids or self.store.ids(s); A = res["alpha"]
        with open(p, "w", encoding="utf-8") as f:
            f.write("id\t" + "\t".join(f"alpha{m + 1}" for m in range(A.shape[1])) + "\tplat\tmaxproto\n")
            for k, i in enumerate(ids):
                f.write(f"{i}\t" + "\t".join(f"{x:.4f}" for x in A[k]) + f"\t{res['plat'][k]:.4f}\t{res['maxproto'][k]:.4f}\n")

    def head_latency(self, tr):
        s = "ar"; idx = torch.arange(min(100, len(self.store.ids(s))))
        Us = tr._inputs(s, idx); S = tr._surf(s, idx); tr.model.eval(); t0 = time.time()
        with torch.no_grad():
            for i in range(len(idx)):
                z, _, _ = tr.model([U[i:i + 1] for U in Us], S[i:i + 1], tr.tau_beta); tr.model.W(z)
        self.costs["head_ms"] = 1000 * (time.time() - t0) / len(idx)

    def tune_diafuse(self, seeds_dev=(0,)):
        grid = list(itertools.product([0.1, 0.3, 0.5], [0.1, 0.5, 1.0], [0.1, 0.5, 1.0]))
        random.Random(0).shuffle(grid)
        best, best_f = -1, {}
        for tau, lcl, lcons in grid[:20]:
            f = {"tau": tau, "lam_cl": lcl, "lam_cons": lcons}
            td = TrainData(self.store, seed=0)
            sc = DiaFuseTrainer(self.store, td, {"fusion": "gate", **f}, 0, self.encs[:3]).fit(["val_ar", "val_gs"]).dev
            print(f"  tune {f}: dev {sc:.4f}", flush=True)
            if sc > best:
                best, best_f = sc, f
        return best_f

    # ------------------------------------------------------------ CEIL
    def run_ceil(self, seeds):
        spec = self.default_spec()
        for seed in seeds:
            if self.done("CEIL", seed):
                continue
            for s in ["ar", "tl", "tz", "na", "my", "myar", "naar", "gsmy"]:
                if not self.store.has(s):
                    continue
                X = self.vecs(s, spec); y = self.store.y(s)
                pred = np.zeros(len(y), int); pr = np.zeros(len(y))
                for tr_i, te_i in StratifiedKFold(5, shuffle=True, random_state=seed).split(X, y):
                    rs = np.random.default_rng(seed); inner = rs.permutation(tr_i); k = len(inner) // 10
                    m = train_pooled(X[inner[k:]], y[inner[k:]], [(X[inner[:k]], y[inner[:k]])], seed)
                    r = pooled_predict(m, X[te_i]); pred[te_i] = r["pred"]; pr[te_i] = r["p_raw"]
                self.write_preds("CEIL", seed, s, {"pred": pred, "p_raw": pr, "p_glob": pr, "p_var": pr})


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--data", default="data"); ap.add_argument("--feats", default="feats")
    ap.add_argument("--out", default="runs"); ap.add_argument("--encoders", default="xlmr,me5,labse")
    ap.add_argument("--mode", default="quick", choices=["quick", "full"])
    ap.add_argument("--only", default=None, help="comma list of system groups to run")
    ap.add_argument("--skip", default="", help="comma list of groups to skip")
    ap.add_argument("--epochs", type=int, default=None)
    args = ap.parse_args()
    if args.epochs:
        HP["epochs"] = args.epochs; HP["p1"] = max(1, round(args.epochs * 0.1)); HP["p2"] = max(HP["p1"] + 1, round(args.epochs * 0.43))
    R = Runner(args)
    seeds_main = list(range(10 if args.mode == "full" else 3))
    seeds_abl = list(range(5 if args.mode == "full" else 2))
    groups = ["B0", "B2", "B3", "B4", "B5", "B6", "B7", "B8", "B9", "B10", "B12", "CEIL", "abl", "abl2", "lt", "fact"]
    run = [g for g in (args.only.split(",") if args.only else groups) if g not in args.skip.split(",")]
    E3 = R.encs[:3]; spec6 = R.default_spec(E3)
    log = lambda *x: print(time.strftime("%H:%M:%S"), *x, flush=True)

    if "B6" in run:
        log("B6 static concatenation"); R.run_pooled("B6", spec6, seeds_main, tune=True)
    if "B0" in run:
        log("B0 Arabic-only / B0-TT")
        R.run_pooled("B0", spec6, seeds_main, train_views=("train_ar",), dev_sets=("val_ar",))
        tt = {s: f"tt-{s}" for s in ["gs", "gu", "tl", "tz", "na", "my", "gsmy"]}
        R.run_pooled("B0TT", spec6, seeds_main, train_views=("train_ar",), dev_sets=("val_ar",), eval_map=tt)
    if "B3" in run or "B2" in run:
        log("B3 single encoders"); dev = {}
        for k, e in enumerate(E3):
            dev[e] = R.run_pooled(f"B3-e{k + 1}", [(e, -1, DEFAULT_POOL[e])], seeds_main)
        best = max(E3, key=lambda e: dev[e] or -1)
        R.run_pooled("B3", [(best, -1, DEFAULT_POOL[best])], seeds_main)
        cands = [e for e in ["me5", "labse", "bgem3"] if e in R.encs]
        b2 = max(cands, key=lambda e: dev.get(e) or -1)
        R.run_pooled("B2", [(b2, -1, DEFAULT_POOL[b2])], seeds_main)
    if "B4" in run:
        log("B4 last-layer mean")
        sc = {e: R.run_pooled(f"B4-{e}", [(e, -1, "mean")], seeds_main[:1]) for e in E3}
        best = max(E3, key=lambda e: sc[e] or -1); R.run_pooled("B4", [(best, -1, "mean")], seeds_main)
    if "B5" in run:
        log("B5 dev-selected layer and pooling (logistic-regression probe)")
        best, cfg = -1, None
        for e in E3:
            for li in range(R.store.feat("val_ar", e).shape[1]):
                for pool in ("mean", "tok"):
                    sp = [(e, li, pool)]
                    Xtr = torch.cat([R.vecs(v, sp) for v in ("train_ar", "train_gs")]).numpy()
                    ytr = np.concatenate([R.store.y(v) for v in ("train_ar", "train_gs")])
                    clf = LogisticRegression(max_iter=300, C=1.0).fit(Xtr, ytr)
                    sc = np.mean([macro_f1(R.store.y(s), clf.predict(R.vecs(s, sp).numpy())) for s in ("val_ar", "val_gs")])
                    if sc > best:
                        best, cfg = sc, sp
        log("  B5 selected", cfg); R.run_pooled("B5", cfg, seeds_main)
    if "B7" in run:
        log("B7 static fusion")
        dims = [R.vecs("val_ar", [s]).shape[1] for s in spec6]
        sc = {m: R.run_pooled(f"B7-{m}", spec6, seeds_main[:1], kind=f"fuse7:{m}", extra={"dims": dims}) for m in ("sum", "hadamard", "residual")}
        best = max(sc, key=lambda m: sc[m] or -1); R.run_pooled("B7", spec6, seeds_main, kind=f"fuse7:{best}", extra={"dims": dims})
    if "B8" in run:
        log("B8 CE+SupCon"); R.run_pooled("B8", spec6, seeds_main, kind="supcon")
    if "B9" in run:
        log("B9 BCL"); R.run_pooled("B9", spec6, seeds_main, kind="bcl", prior_la=True)
    b10_flags = {"fusion": "single", "proto": "class", "cells": "class", "hardness": "proto", "cons": False, "cal": False}
    if "B10" in run:
        log("B10 SharpReCL-style")
        R.run_diafuse("B10", seeds_main, b10_flags, encs=E3, input_mode="pooled", variety_wise=False)
    best12 = {}
    if "B12" in run:
        if args.mode == "full":
            log("B12 tuning (20 configurations)"); best12 = R.tune_diafuse()
            json.dump(best12, open(os.path.join(args.out, "b12_hparams.json"), "w"))
        log("B12 DiaFuse-PRC", best12)
        R.run_diafuse("B12", seeds_main, best12, diag=True, also_nopc=True)
    elif os.path.exists(os.path.join(args.out, "b12_hparams.json")):
        best12 = json.load(open(os.path.join(args.out, "b12_hparams.json")))
    if "CEIL" in run:
        log("CEIL target-supervised ceiling"); R.run_ceil(seeds_main)
    if "abl" in run:
        log("ablations")
        A = {"abl-layer": {"layer_mode": "last"}, "abl-static": {"fusion": "static"}, "abl-nos": {"gate_input": "e"},
             "abl-protocls": {"proto": "class"}, "abl-protonone": {"proto": "none"}, "abl-hm": {"hm": False},
             "abl-hmproto": {"hardness": "proto"}, "abl-cons": {"cons": False}, "abl-cal": {"cal": False}}
        for name, f in A.items():
            R.run_diafuse(name, seeds_abl, {**best12, **f}, variety_wise=name != "abl-cal")
        R.run_diafuse("abl-nosyn", seeds_abl, {**best12, "cons": False}, data_kw={"arabic_only": True})
        if R.store.has("train_madar_tn"):
            R.run_diafuse("abl-xling", seeds_abl, {**best12, "xling": True})
        else:
            log("  abl-xling skipped: no MADAR pairs (set MADAR_DIR before prep_data.py)")
        sc = {e: R.run_diafuse(f"abl-single-{e}", seeds_abl[:1], {**best12, "fusion": "single"}, encs=[e]) for e in E3}
        best = max(E3, key=lambda e: sc[e] or -1)
        R.run_diafuse("abl-single", seeds_abl, {**best12, "fusion": "single"}, encs=[best])
    if "abl2" in run:
        log("additional ablations")
        A2 = {"abl2-gatecons": {"gatecons": True}, "abl2-sonly": {"gate_input": "s"}, "abl2-mean": {"layer_mode": "mean"},
              "abl2-tok": {"layer_mode": "tok"}, "abl2-indepA": {"indepA": True}, "abl2-k0": {"kappa": 0.0},
              "abl2-k10": {"kappa": 10.0}, "abl2-kinf": {"kappa": float("inf")}, "abl2-hdist": {"hardness": "dist"},
              "abl2-hent": {"hardness": "ent"}, "abl2-hmar": {"hardness": "mar"}, "abl2-consemb": {"cons_terms": "emb"},
              "abl2-conspred": {"cons_terms": "pred"}}
        for name, f in A2.items():
            R.run_diafuse(name, seeds_abl, {**best12, **f})
        R.run_diafuse("abl2-tronly", seeds_abl, best12, data_kw={"families": ("TR",)})
        R.run_diafuse("abl2-twogen", seeds_abl, best12, data_kw={"twogen": True})
        if len(E3) == 3:
            R.run_diafuse("abl2-m2", seeds_abl, best12, encs=E3[:2])
        if "marbert" in R.encs:
            R.run_diafuse("abl2-m4", seeds_abl, best12, encs=E3 + ["marbert"])
    if "lt" in run:
        log("controlled imbalance")
        for irc, irv in ((10, 1), (1, 4), (1, 10), (10, 10)):
            dk = {"ir_c": irc if irc > 1 else None, "r_syn": 1.0 / irv}
            tag = f"lt-{irc}-{irv}"
            R.run_diafuse(f"{tag}-B12", seeds_main, best12, data_kw=dk)
            R.run_diafuse(f"{tag}-B10", seeds_main, b10_flags, data_kw=dk, input_mode="pooled", variety_wise=False)
            for sysname, kind, la in (("B8", "supcon", False), ("B9", "bcl", True)):
                for seed in seeds_main:
                    if R.done(f"{tag}-{sysname}", seed):
                        continue
                    td = TrainData(R.store, seed=seed, **dk)
                    XA, XL = R.vecs("train_ar", spec6), R.vecs("train_gs", spec6)
                    rows = torch.tensor([i for _, i in td.items]); isl = torch.tensor([v != "AR" for v, _ in td.items])
                    Xtr = torch.where(isl[:, None], XL[rows], XA[rows])
                    m = train_pooled(Xtr, td.label, [(R.vecs(s, spec6), R.store.y(s)) for s in ("val_ar", "val_gs")], seed, kind, None, None, la)
                    for s in R.eval_sets:
                        R.write_preds(f"{tag}-{sysname}", seed, s, pooled_predict(m, R.vecs(s, spec6)))
    if "fact" in run:
        log("factorial 2^4")
        for bits in itertools.product([0, 1], repeat=4):
            fu, pr, hm, co = bits
            f = {**best12, "fusion": "gate" if fu else "static", "proto": "hier" if pr else "none", "hm": bool(hm), "cons": bool(co)}
            R.run_diafuse("fact-" + "".join(map(str, bits)), seeds_abl, f)

    # costs for Table 17
    man_p = os.path.join(args.out, "manual.json")
    man = json.load(open(man_p)) if os.path.exists(man_p) else {}
    meta = {e: R.store.meta[e] for e in R.encs}
    ext = {e: meta[e].get("sec_per_1k") for e in E3}
    lat = {e: meta[e].get("latency_ms") for e in E3}
    if all(ext.values()):
        man["cost/B3/extract"] = f"{min(ext.values()):.1f}"
        for s in ("B6", "B10", "B12"):
            man[f"cost/{s}/extract"] = f"{sum(ext.values()):.1f}"
    for s in ("B3", "B6", "B10", "B12"):
        if s in R.costs:
            man[f"cost/{s}/train"] = f"{np.mean(R.costs[s]):.1f}"
    if all(lat.values()):
        head = R.costs.get("head_ms", 0.0)
        man["cost/B3/infer"] = f"{min(lat.values()) + head:.1f}"
        for s in ("B6", "B10", "B12"):
            man[f"cost/{s}/infer"] = f"{sum(lat.values()) + head:.1f}"
    for s_ in ("B10", "B12"):
        if f"{s_}/params" in R.costs:
            man[f"cost/{s_}/params"] = f"{R.costs[f'{s_}/params'] / 1e6:.2f}M"
    qmeta = os.path.join(args.feats, "qwen", "meta.json")
    if os.path.exists(qmeta) and all(ext.values()) and all(lat.values()):
        q = json.load(open(qmeta))
        man["cost/qwen/extract"] = f"{sum(ext.values()) + q['sec_per_1k']:.1f}"
        man["cost/qwen/infer"] = f"{sum(lat.values()) + q['latency_ms'] + R.costs.get('head_ms', 0.0):.1f}"
        shapes = [(R.store.feat("val_ar", e).shape[1], R.store.feat("val_ar", e).shape[2]) for e in E3]
        shapes.append((len(q["layers"]), 2 * q["dim"]))
        man["cost/qwen/params"] = f"{sum(p.numel() for p in DiaFuse(shapes, {'fusion': 'gate'}).parameters()) / 1e6:.2f}M"
        man["cost/qwen/train"] = "n/m"
    json.dump(man, open(man_p, "w"), indent=1)
    log("done")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile finetune.py
#!/usr/bin/env python3
"""Fine-tuned baselines (GPU recommended).

B1 : XLM-R-large fully fine-tuned with cross-entropy.
B11: robustness/calibration baseline in the style of Shao et al. -- attention pooling,
     l2 consistency between clean and transformed views, post-hoc temperature scaling.
Both are trained on TSAC-AR train + its G_seen view and selected on val_ar + val_gs, and
write preds/<system>/<seed>/<set>.tsv in the format of diafuse_eval.py.

    python finetune.py --system B1 --seeds 0,1,2 --model models/xlmr
"""
import argparse
import copy
import csv
import json
import math
import os
import random
import sys
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from translit import script_of  # noqa: E402

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def read(path):
    with open(path, encoding="utf-8", newline="") as f:
        rows = list(csv.DictReader(f, delimiter="\t"))
    return [r["id"] for r in rows], [r["text"] for r in rows], np.array([int(r["label"]) for r in rows])


class Clf(nn.Module):
    def __init__(self, src, attention_pool):
        super().__init__()
        self.enc = AutoModel.from_pretrained(src)
        h = self.enc.config.hidden_size
        self.attn = attention_pool
        self.q = nn.Parameter(torch.randn(h) / math.sqrt(h)) if attention_pool else None
        self.drop = nn.Dropout(0.1); self.out = nn.Linear(h, 2)

    def forward(self, **enc):
        H = self.enc(**enc).last_hidden_state
        if self.attn:
            s = (H @ self.q).masked_fill(enc["attention_mask"] == 0, -1e4)
            a = torch.softmax(s, -1)
            v = torch.einsum("bt,bth->bh", a, H)
        else:
            v = H[:, 0]
        return self.out(self.drop(v)), v


def macro_f1(y, p):
    f = []
    for c in (0, 1):
        tp = np.sum((p == c) & (y == c)); fp = np.sum((p == c) & (y != c)); fn = np.sum((p != c) & (y == c))
        f.append(0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn))
    return float(np.mean(f))


@torch.no_grad()
def logits_of(model, tok, texts, max_len, bs=64):
    model.eval(); out = []
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i + bs], padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(DEV)
        with torch.autocast(device_type=DEV.type, dtype=torch.float16, enabled=DEV.type == "cuda"):
            out.append(model(**enc)[0].float())
    return torch.cat(out)


def fit_T(logits, y):
    y = torch.tensor(y, device=logits.device); best, bT = 1e9, 1.0
    for T in np.exp(np.linspace(np.log(0.05), np.log(20), 120)):
        v = F.cross_entropy(logits / float(T), y).item()
        if v < best:
            best, bT = v, float(T)
    return bT


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--system", required=True, choices=["B1", "B11"])
    ap.add_argument("--model", default="FacebookAI/xlm-roberta-large")
    ap.add_argument("--data", default="data"); ap.add_argument("--out", default="runs")
    ap.add_argument("--seeds", default="0,1,2"); ap.add_argument("--epochs", type=int, default=3)
    ap.add_argument("--lr", type=float, default=1e-5); ap.add_argument("--batch", type=int, default=16)
    ap.add_argument("--max_len", type=int, default=64); ap.add_argument("--lam", type=float, default=0.5)
    args = ap.parse_args()
    D = args.data
    tr_ids, ar_txt, ytr = read(f"{D}/train_ar.tsv")
    views = {v: read(f"{D}/train_{v}.tsv")[1] for v in ("gs", "inf", "noise", "cs") if os.path.exists(f"{D}/train_{v}.tsv")}
    train = [(t, int(y), i, "ar") for i, (t, y) in enumerate(zip(ar_txt, ytr))] + \
            [(t, int(y), i, "gs") for i, (t, y) in enumerate(zip(views["gs"], ytr))]
    dev = [read(f"{D}/{s}.tsv") for s in ("val_ar", "val_gs")]
    eval_sets = sorted(f[:-4] for f in os.listdir(D) if f.endswith(".tsv") and not f.startswith(("train", "val", "tt-")))
    tok = AutoTokenizer.from_pretrained(args.model)
    costs = []
    for seed in [int(s) for s in args.seeds.split(",")]:
        if os.path.exists(os.path.join(args.out, "preds", args.system, str(seed), "my.tsv")):
            continue
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        model = Clf(args.model, args.system == "B11").to(DEV)
        params = [{"params": model.enc.parameters(), "lr": args.lr},
                  {"params": [p for n, p in model.named_parameters() if not n.startswith("enc.")], "lr": 1e-3}]
        opt = torch.optim.AdamW(params, weight_decay=0.01)
        bs = args.batch if args.system == "B1" else max(1, args.batch // 2)
        steps = args.epochs * math.ceil(len(train) / bs)
        sch = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
        scaler = torch.amp.GradScaler(enabled=DEV.type == "cuda")
        best, best_state = -1, None; t0 = time.time()
        for ep in range(args.epochs):
            model.train(); order = np.random.permutation(len(train))
            for b in range(0, len(order), bs):
                batch = [train[i] for i in order[b:b + bs]]
                texts = [x[0] for x in batch]; y = torch.tensor([x[1] for x in batch], device=DEV)
                with torch.autocast(device_type=DEV.type, dtype=torch.float16, enabled=DEV.type == "cuda"):
                    enc = tok(texts, padding=True, truncation=True, max_length=args.max_len, return_tensors="pt").to(DEV)
                    logits, h = model(**enc)
                    loss = F.cross_entropy(logits.float(), y)
                    if args.system == "B11":  # l2 consistency with a transformed view of the same sentence
                        partners = []
                        for t, _, i, v in batch:
                            opts = ["ar"] if v != "ar" else []
                            opts += [k for k in views if k != v]
                            k = random.choice(opts)
                            partners.append(ar_txt[i] if k == "ar" else views[k][i])
                        enc2 = tok(partners, padding=True, truncation=True, max_length=args.max_len, return_tensors="pt").to(DEV)
                        logits2, h2 = model(**enc2)
                        loss = loss + F.cross_entropy(logits2.float(), y) + args.lam * ((h.float() - h2.float()) ** 2).sum(-1).mean() / h.shape[-1]
                opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step()
            sc = np.mean([macro_f1(y, logits_of(model, tok, t, args.max_len).argmax(-1).cpu().numpy()) for _, t, y in dev])
            print(f"[{args.system}] seed {seed} epoch {ep}: dev macro-F1 {sc:.4f}", flush=True)
            if sc > best:
                best = sc; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        model.load_state_dict(best_state); costs.append((time.time() - t0) / 60)
        Ld = [logits_of(model, tok, t, args.max_len) for _, t, _ in dev]
        T = fit_T(torch.cat(Ld), np.concatenate([y for _, _, y in dev]))
        Tv = {}
        for k, (L, (_, _, y)) in enumerate(zip(Ld, dev)):
            Tk = fit_T(L, y); n = len(y)
            Tv[k] = math.exp((n * math.log(Tk) + 100 * math.log(T)) / (n + 100))
        for s in eval_sets:
            ids, texts, _ = read(f"{D}/{s}.tsv")
            L = logits_of(model, tok, texts, args.max_len)
            vt = torch.tensor([Tv[1] if script_of(t) == "lat" else Tv[0] for t in texts], device=DEV)[:, None]
            p_raw = torch.softmax(L, -1)[:, 1]; p_glob = torch.softmax(L / T, -1)[:, 1]; p_var = torch.softmax(L / vt, -1)[:, 1]
            pred = L.argmax(-1)
            p = os.path.join(args.out, "preds", args.system, str(seed), f"{s}.tsv"); os.makedirs(os.path.dirname(p), exist_ok=True)
            with open(p, "w", encoding="utf-8") as f:
                f.write("id\tpred\tp_raw\tp_glob\tp_var\n")
                for i, a, b, c, d in zip(ids, pred.tolist(), p_raw.tolist(), p_glob.tolist(), p_var.tolist()):
                    f.write(f"{i}\t{a}\t{b:.6f}\t{c:.6f}\t{d:.6f}\n")
        del model; torch.cuda.empty_cache() if torch.cuda.is_available() else None
    if costs:
        mp = os.path.join(args.out, "manual.json"); man = json.load(open(mp)) if os.path.exists(mp) else {}
        man[f"cost/{args.system}/train"] = f"{np.mean(costs):.1f}"
        json.dump(man, open(mp, "w"), indent=1)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile diafuse_eval.py
#!/usr/bin/env python3
"""DiaFuse-PRC evaluation pipeline.

Reads real prediction files and writes results/results.tex (one
\\setres{system/set/metric}{value} per computed quantity) plus the data files of
the figures. Nothing is invented: a quantity whose inputs are missing is simply
not written, and the manuscript then prints TBO for it.

Usage
-----
    python diafuse_eval.py --data DATA --preds PREDS [--diag DIAG] [--emb EMB]
                           [--audit AUDIT] [--manual manual.json] [--out results]

Layout (all files are UTF-8 TSV with a header row)
--------------------------------------------------
DATA/<set>.tsv            id, text, label (0 = negative, 1 = positive)
    sets: train_ar, train_gs, val_ar, ar, gs, gu, tl, tz, na, my, naar, myar,
          gsmy, rob-<family> (severity 2), rob-tru-s0 ... rob-tru-s4
    gs/gu/rob-trs/rob-tru*/ share ids with ar; other rob-* share ids with gs;
    naar shares ids with na.
PREDS/<system>/<seed>/<set>.tsv   id, pred [, p_raw, p_glob, p_var]
    p_* = probability of the positive class under each calibration variant.
    systems: B0, B0TT, B1 ... B12, B12L, CEIL, B12-nopc, abl-*, abl2-*,
             lt-<irC>-<irV>-<sys>, fact-<FPHC bits>, B3-e1/B3-e2/B3-e3
DIAG/<system>/<seed>/<set>.tsv    id, alpha1..alphaM, plat, maxproto
    (also <set> = val for the off-support threshold)
EMB/<system>/<set>.npy            fused embeddings z, rows aligned with DATA/<set>.tsv
                                  (needs train_gs.npy for PAD)
AUDIT/gen_gs.tsv, gen_gu.tsv      id, ann1, ann2 [, final]   (1 = acceptable)
AUDIT/errors_tz.tsv, errors_my.tsv id, ann1, ann2, final     (E1 ... E6)
manual.json                       {"key": "value"} copied verbatim (counts, timings)

Outputs (in --out): results.tex, ci.csv (95% CIs of macro-F1), summary.json,
factorial.json, chain.dat, rel-B6-glob.dat, rel-B12-var.dat, robcurve-tru.dat,
umap-B6.dat, umap-B12.dat (each only when its inputs exist).
"""
import argparse
import csv
import json
import math
import os
import re
import sys
from collections import Counter, defaultdict

import numpy as np

RNG = np.random.default_rng(12345)
N_BOOT = 1000

CHAIN = ["ar", "gs", "gu", "tl", "tz", "na", "my"]
HUMAN = ["tl", "tz", "na", "my"]
ARABIC_SETS = {"train_ar", "val_ar", "ar", "naar", "myar"}
PAIR = {"gs": "ar", "gu": "ar", "na": "naar"}  # EM(S, PAIR[S])
MAIN = ["B0", "B0TT", "B1", "B2", "B3", "B4", "B5", "B6", "B7", "B8", "B9",
        "B10", "B11", "B12", "B12L"]
OURS = {"B12", "B12L"}
LOWER_BETTER = {"ece_raw", "ece_glob", "ece_var", "brier", "nll", "aurc"}

# Dialect-marker lexicons (Appendix E). Native-speaker verification required.
MARKERS = {
    "tn": {"barcha", "barsha", "behi", "bahi", "tawa", "taw", "chnowa", "chnoua",
           "chniya", "mouch", "moch", "yesser", "yasser", "mte3", "mta3", "famma",
           "برشا", "باهي", "توا", "شنوة", "شنية", "موش", "ياسر", "متاع", "فما"},
    "ma": {"bzzaf", "bzaf", "mzyan", "mezyan", "daba", "achno", "chno", "dyal",
           "dial", "bghit", "wakha", "safi", "zwin",
           "بزاف", "مزيان", "دابا", "اشنو", "ديال", "بغيت", "واخا", "صافي", "زوين"},
    "dz": {"mlih", "drok", "dorka", "ta3", "nta3", "chbab",
           "مليح", "درك", "تاع", "نتاع"},
}
DIGIT_LETTERS = set("235789")


# ----------------------------------------------------------------- utilities
def read_tsv(path):
    with open(path, encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f, delimiter="\t"))


def load_set(data_dir, name):
    p = os.path.join(data_dir, f"{name}.tsv")
    if not os.path.exists(p):
        return None
    rows = read_tsv(p)
    return {"id": [r["id"] for r in rows], "text": [r["text"] for r in rows],
            "y": np.array([int(r["label"]) for r in rows])}


def f1_macro(y, p):
    out = []
    for c in (0, 1):
        tp = np.sum((p == c) & (y == c)); fp = np.sum((p == c) & (y != c))
        fn = np.sum((p != c) & (y == c))
        out.append(0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn))
    return float(np.mean(out))


def f1_class(y, p, c):
    tp = np.sum((p == c) & (y == c)); fp = np.sum((p == c) & (y != c))
    fn = np.sum((p != c) & (y == c))
    return 0.0 if tp == 0 else float(2 * tp / (2 * tp + fp + fn))


def conf_correct(y, pred, ppos):
    conf = np.where(pred == 1, ppos, 1 - ppos)
    return conf, (pred == y).astype(float)


def ece_equal_mass(conf, corr, bins=15):
    order = np.argsort(conf)
    chunks = np.array_split(order, bins)
    n = len(conf)
    return float(sum(len(c) / n * abs(conf[c].mean() - corr[c].mean()) for c in chunks if len(c)))


def reliability(conf, corr, bins=15):
    order = np.argsort(conf)
    return [(float(conf[c].mean()), float(corr[c].mean())) for c in np.array_split(order, bins) if len(c)]


def aurc(conf, corr):
    order = np.argsort(-conf)
    err = 1 - corr[order]
    return float(np.mean(np.cumsum(err) / np.arange(1, len(err) + 1)))


def strat_boot_idx(y):
    idx = []
    for c in np.unique(y):
        pos = np.flatnonzero(y == c)
        idx.append(RNG.choice(pos, size=len(pos), replace=True))
    return np.concatenate(idx)


def cohen_kappa(a, b):
    a = list(a); b = list(b)
    cats = sorted(set(a) | set(b)); n = len(a)
    po = sum(x == y for x, y in zip(a, b)) / n
    ca, cb = Counter(a), Counter(b)
    pe = sum(ca[c] * cb[c] for c in cats) / n ** 2
    return 1.0 if pe == 1 else (po - pe) / (1 - pe)


# ------------------------------------------------------------- formatting
class Results:
    def __init__(self):
        self.vals = {}

    def set(self, key, value):
        self.vals[key] = value

    def write(self, path):
        with open(path, "w", encoding="utf-8") as f:
            f.write("% Generated by diafuse_eval.py from prediction files. Do not edit by hand.\n")
            for k in sorted(self.vals):
                f.write(f"\\setres{{{k}}}{{{self.vals[k]}}}\n")


def fmt(x, d=1):
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return "--"
    out = f"{x:.{d}f}"
    return out[1:] if re.fullmatch(r"-0\.0*", out) else out


def fmt_ci(m, lo, hi, d=1):
    return f"{fmt(m, d)}{{\\tiny\\,[{fmt(lo, d)}, {fmt(hi, d)}]}}"


def fmt_int(n):
    return f"{int(n):,}"


SCALE = {"mf1": (100, 1), "acc": (100, 1), "minf1": (100, 1), "wga": (100, 1),
         "em": (100, 1), "ece_raw": (100, 1), "ece_glob": (100, 1), "ece_var": (100, 1),
         "brier": (1, 3), "nll": (1, 3), "aurc": (100, 1), "mf1min": (100, 1)}


# ------------------------------------------------------------- predictions
def load_preds(pred_dir, data):
    """preds[system][seed][set] = dict(pred=..., p_raw/p_glob/p_var=...), aligned to data ids."""
    preds = {}
    if not pred_dir or not os.path.isdir(pred_dir):
        return preds
    for system in sorted(os.listdir(pred_dir)):
        sdir = os.path.join(pred_dir, system)
        if not os.path.isdir(sdir):
            continue
        for seed in sorted(os.listdir(sdir)):
            ddir = os.path.join(sdir, seed)
            if not os.path.isdir(ddir):
                continue
            for fn in os.listdir(ddir):
                if not fn.endswith(".tsv"):
                    continue
                name = fn[:-4]
                if name not in data:
                    print(f"[warn] {system}/{seed}/{fn}: no DATA/{name}.tsv, skipped", file=sys.stderr)
                    continue
                rows = {r["id"]: r for r in read_tsv(os.path.join(ddir, fn))}
                ids = data[name]["id"]
                missing = [i for i in ids if i not in rows]
                if missing:
                    print(f"[warn] {system}/{seed}/{fn}: {len(missing)} ids missing, skipped", file=sys.stderr)
                    continue
                rec = {"pred": np.array([int(rows[i]["pred"]) for i in ids])}
                for col in ("p_raw", "p_glob", "p_var"):
                    if col in rows[ids[0]] and rows[ids[0]][col] != "":
                        rec[col] = np.clip(np.array([float(rows[i][col]) for i in ids]), 1e-7, 1 - 1e-7)
                preds.setdefault(system, {}).setdefault(seed, {})[name] = rec
    return preds


def seeds_with(preds, system, sets):
    return [s for s, d in preds.get(system, {}).items() if all(x in d for x in sets)]


def per_seed_metric(preds, data, system, name, metric, minority=None):
    vals = []
    for seed in seeds_with(preds, system, [name]):
        r = preds[system][seed][name]; y = data[name]["y"]
        if metric == "mf1":
            vals.append(f1_macro(y, r["pred"]))
        elif metric == "acc":
            vals.append(float(np.mean(r["pred"] == y)))
        elif metric == "minf1" and minority is not None:
            vals.append(f1_class(y, r["pred"], minority))
        elif metric in ("ece_raw", "ece_glob", "ece_var"):
            col = "p_" + metric.split("_")[1]
            if col in r:
                vals.append(ece_equal_mass(*conf_correct(y, r["pred"], r[col])))
        elif metric in ("brier", "nll", "aurc"):
            col = next((c for c in ("p_var", "p_glob", "p_raw") if c in r), None)
            if col is None:
                continue
            p = r[col]
            if metric == "brier":
                vals.append(float(np.mean((p - y) ** 2)))
            elif metric == "nll":
                vals.append(float(-np.mean(np.log(np.where(y == 1, p, 1 - p)))))
            else:
                vals.append(aurc(*conf_correct(y, r["pred"], p)))
    return vals


def boot_mf1(preds, data, system, name, n=N_BOOT):
    """Class-stratified bootstrap within each seed, pooled across seeds."""
    y = data[name]["y"]; out = []
    seeds = seeds_with(preds, system, [name])
    per = max(1, n // max(1, len(seeds)))
    for seed in seeds:
        p = preds[system][seed][name]["pred"]
        for _ in range(per):
            idx = strat_boot_idx(y)
            out.append(f1_macro(y[idx], p[idx]))
    return np.array(out)


def paired_boot_p(preds, data, sys_a, sys_b, name, n=N_BOOT):
    """One-sided p for mean macro-F1(A) > mean macro-F1(B), seed-averaged, same resamples."""
    y = data[name]["y"]
    sa = seeds_with(preds, sys_a, [name]); sb = seeds_with(preds, sys_b, [name])
    if not sa or not sb:
        return None
    pa = [preds[sys_a][s][name]["pred"] for s in sa]
    pb = [preds[sys_b][s][name]["pred"] for s in sb]
    worse = 0
    for _ in range(n):
        idx = strat_boot_idx(y)
        d = np.mean([f1_macro(y[idx], p[idx]) for p in pa]) - np.mean([f1_macro(y[idx], p[idx]) for p in pb])
        worse += d <= 0
    return (worse + 1) / (n + 1)


def holm(pvals):
    items = sorted([(p, k) for k, p in pvals.items() if p is not None])
    m = len(items); adj = {}; running = 0.0
    for i, (p, k) in enumerate(items):
        running = max(running, min(1.0, (m - i) * p))
        adj[k] = running
    return adj


# ------------------------------------------------------------- main metrics
def classification_metrics(R, preds, data, minority):
    summary = {}
    for system in preds:
        for name in list(data):
            if name.startswith(("train", "val", "tt-")):
                continue
            for metric in ("mf1", "acc", "ece_raw", "ece_glob", "ece_var", "brier", "nll", "aurc"):
                v = per_seed_metric(preds, data, system, name, metric)
                if not v:
                    continue
                k, d = SCALE[metric]
                m = float(np.mean(v)); sd = float(np.std(v, ddof=1)) if len(v) > 1 else 0.0
                summary[(system, name, metric)] = (m, sd, v)
                R.set(f"{system}/{name}/{metric}:m", fmt(k * m, d))
                R.set(f"{system}/{name}/{metric}", f"{fmt(k * m, d)}\\pmsd{{{fmt(k * sd, d)}}}")
                if metric == "mf1":
                    R.set(f"{system}/{name}/mf1min:m", fmt(100 * min(v), 1))
            # cross-script exact match
            pair = PAIR.get(name) or ("ar" if name in ("rob-trs", "rob-tru") or name.startswith("rob-tru-s")
                                      else ("gs" if name.startswith("rob-") else None))
            if pair and pair in data:
                ems = []
                for seed in seeds_with(preds, system, [name, pair]):
                    a = dict(zip(data[name]["id"], preds[system][seed][name]["pred"]))
                    b = dict(zip(data[pair]["id"], preds[system][seed][pair]["pred"]))
                    common = [i for i in a if i in b]
                    if common:
                        ems.append(np.mean([a[i] == b[i] for i in common]))
                if ems:
                    summary[(system, name, "em")] = (float(np.mean(ems)), 0.0, ems)
                    R.set(f"{system}/{name}/em:m", fmt(100 * np.mean(ems)))
        # aggregates: human chain, drop, robustness consistency
        for metric in ("mf1", "acc", "ece_raw", "ece_glob", "ece_var", "brier", "nll", "aurc"):
            if all((system, s, metric) in summary for s in HUMAN):
                seeds = seeds_with(preds, system, HUMAN)
                if not seeds:
                    continue
                per_seed = []
                for sd in seeds:
                    vals = [per_seed_metric({system: {sd: preds[system][sd]}}, data, system, s, metric) for s in HUMAN]
                    if all(vals):
                        per_seed.append(float(np.mean([v[0] for v in vals])))
                if not per_seed:
                    continue
                k, d = SCALE[metric]
                m = float(np.mean(per_seed)); sdv = float(np.std(per_seed, ddof=1)) if len(per_seed) > 1 else 0.0
                summary[(system, "hum", metric)] = (m, sdv, per_seed)
                R.set(f"{system}/hum/{metric}:m", fmt(k * m, d))
                R.set(f"{system}/hum/{metric}", f"{fmt(k * m, d)}\\pmsd{{{fmt(k * sdv, d)}}}")
                if metric == "mf1":
                    R.set(f"{system}/hum/mf1min:m", fmt(100 * min(per_seed)))
        seeds = seeds_with(preds, system, ["ar", "my"])
        if seeds:
            drops = [f1_macro(data["ar"]["y"], preds[system][s]["ar"]["pred"]) -
                     f1_macro(data["my"]["y"], preds[system][s]["my"]["pred"]) for s in seeds]
            sdv = float(np.std(drops, ddof=1)) if len(drops) > 1 else 0.0
            R.set(f"{system}/drop/mf1", f"{fmt(100 * np.mean(drops))}\\pmsd{{{fmt(100 * sdv)}}}")
        rob_em = [summary[(system, n, "em")][0] for n in data
                  if n.startswith("rob-") and "-s" not in n[4:] and (system, n, "em") in summary]
        if rob_em:
            R.set(f"{system}/rob/em:m", fmt(100 * np.mean(rob_em)))
        # worst-group accuracy and minority-class F1 on S0 u S3
        seeds = seeds_with(preds, system, ["ar", "tl"])
        if seeds:
            wgas, minf = [], []
            for s in seeds:
                cells = []
                for name in ("ar", "tl"):
                    y = data[name]["y"]; p = preds[system][s][name]["pred"]
                    cells += [np.mean(p[y == c] == c) for c in (0, 1) if np.any(y == c)]
                wgas.append(min(cells))
                y = np.concatenate([data["ar"]["y"], data["tl"]["y"]])
                p = np.concatenate([preds[system][s]["ar"]["pred"], preds[system][s]["tl"]["pred"]])
                minf.append(f1_class(y, p, minority))
            R.set(f"{system}/artl/wga:m", fmt(100 * np.mean(wgas)))
            R.set(f"{system}/artl/minf1:m", fmt(100 * np.mean(minf)))
    return summary


def bold_and_significance(R, preds, data, summary):
    """Bold best non-reference system per chain column; dagger for B12/B12L under the decision rule."""
    cols = CHAIN + ["hum"]
    for name in cols:
        cands = {s: summary[(s, name, "mf1")] for s in MAIN if (s, name, "mf1") in summary}
        if not cands:
            continue
        best = max(cands, key=lambda s: cands[s][0])
        k = f"{best}/{name}/mf1"
        R.set(k, f"\\textbf{{{R.vals[k].split(chr(92) + 'pmsd')[0]}}}\\pmsd" + R.vals[k].split("\\pmsd")[1])
    for ours in OURS:
        pv, diffs = {}, {}
        for name in CHAIN:
            if (ours, name, "mf1") not in summary:
                continue
            base = {s: summary[(s, name, "mf1")] for s in MAIN if s not in OURS and (s, name, "mf1") in summary}
            if not base:
                continue
            b = max(base, key=lambda s: base[s][0])
            pv[name] = paired_boot_p(preds, data, ours, b, name, n=300)
            ma, sa, _ = summary[(ours, name, "mf1")]; mb, sb, _ = base[b]
            diffs[name] = (ma - mb, max(sa, sb))
        for name, padj in holm(pv).items():
            diff, sdmax = diffs[name]
            if padj < 0.05 and diff > sdmax:
                k = f"{ours}/{name}/mf1"
                head, tail = R.vals[k].split("\\pmsd")
                R.set(k, f"{head}\\sig\\pmsd{tail}")


def decomposition(R, preds, data, n=N_BOOT):
    need = ["ar", "tz", "myar", "my"]
    if not all(s in data for s in need):
        return
    ceil = {}
    for s in need:
        v = per_seed_metric(preds, data, "CEIL", s, "mf1")
        if v:
            ceil[s] = float(np.mean(v))

    def mean_f1(system, name, idx=None):
        y = data[name]["y"]
        vals = []
        for seed in seeds_with(preds, system, [name]):
            p = preds[system][seed][name]["pred"]
            vals.append(f1_macro(y[idx], p[idx]) if idx is not None else f1_macro(y, p))
        return float(np.mean(vals))

    def effects(F):
        a, b, c, d = F["ar"], F["tz"], F["myar"], F["my"]
        return {"script": 0.5 * ((a - b) + (c - d)), "dial": 0.5 * ((a - c) + (b - d)), "int": (c - d) - (a - b)}

    for system in preds:
        if not seeds_with(preds, system, need):
            continue
        F = {s: mean_f1(system, s) for s in need}
        point = effects(F)
        boots = defaultdict(list); boots_rho = defaultdict(list)
        for _ in range(n // 4):
            Fb = {s: mean_f1(system, s, strat_boot_idx(data[s]["y"])) for s in need}
            for k, v in effects(Fb).items():
                boots[k].append(v)
            if len(ceil) == 4:
                for k, v in effects({s: Fb[s] / ceil[s] for s in need}).items():
                    boots_rho[k].append(v)
        for k in ("script", "dial", "int"):
            lo, hi = np.percentile(boots[k], [2.5, 97.5])
            R.set(f"{system}/dec/{k}", fmt_ci(100 * point[k], 100 * lo, 100 * hi))
        R.set(f"{system}/dec/total", fmt(100 * (F["ar"] - F["my"])))
        if len(ceil) == 4:
            pr = effects({s: F[s] / ceil[s] for s in need})
            for k in ("script", "dial", "int"):
                lo, hi = np.percentile(boots_rho[k], [2.5, 97.5])
                R.set(f"{system}/decrho/{k}", fmt_ci(100 * pr[k], 100 * lo, 100 * hi))
        for key, (s1, s2) in {"plat/effect": ("tl", "tz"), "dz/script": ("naar", "na")}.items():
            if s1 in data and s2 in data and seeds_with(preds, system, [s1, s2]):
                pt = mean_f1(system, s1) - mean_f1(system, s2)
                bs = [mean_f1(system, s1, strat_boot_idx(data[s1]["y"])) -
                      mean_f1(system, s2, strat_boot_idx(data[s2]["y"])) for _ in range(n // 4)]
                lo, hi = np.percentile(bs, [2.5, 97.5])
                R.set(f"{system}/{key}", fmt_ci(100 * pt, 100 * lo, 100 * hi))


# ------------------------------------------------------------- model-free diagnostics
TOKEN_RE = re.compile(r"[^\W_]+", re.UNICODE)
URL_RE = re.compile(r"https?://\S+|www\.\S+|@\w+")


def tokens(text):
    return TOKEN_RE.findall(URL_RE.sub(" ", text.lower()))


def char_ngrams(texts, n=3):
    c = Counter()
    for t in texts:
        s = " " + " ".join(tokens(t)) + " "
        c.update(s[i:i + n] for i in range(len(s) - n + 1))
    return c


def jsd(c1, c2):
    keys = set(c1) | set(c2); n1 = sum(c1.values()); n2 = sum(c2.values())
    p = np.array([c1[k] / n1 for k in keys]); q = np.array([c2[k] / n2 for k in keys])
    m = 0.5 * (p + q)

    def kl(a, b):
        mask = a > 0
        return float(np.sum(a[mask] * np.log2(a[mask] / b[mask])))
    return 0.5 * kl(p, m) + 0.5 * kl(q, m)


def ngrams_of(toks):
    return toks + [a + " " + b for a, b in zip(toks, toks[1:])]


def polarity_cues(texts, y, k=100, min_count=5, a0=1000.0):
    """Weighted log-odds with informative Dirichlet prior (Monroe et al., 2008)."""
    cnt = [Counter(), Counter()]
    for t, lab in zip(texts, y):
        cnt[lab].update(ngrams_of(tokens(t)))
    tot = cnt[0] + cnt[1]; N = sum(tot.values())
    n = [sum(cnt[0].values()), sum(cnt[1].values())]
    z = {}
    for w, c in tot.items():
        if c < min_count:
            continue
        aw = a0 * c / N
        l1 = math.log((cnt[1][w] + aw) / (n[1] + a0 - cnt[1][w] - aw))
        l0 = math.log((cnt[0][w] + aw) / (n[0] + a0 - cnt[0][w] - aw))
        z[w] = (l1 - l0) / math.sqrt(1 / (cnt[1][w] + aw) + 1 / (cnt[0][w] + aw))
    ranked = sorted(z, key=z.get)
    return {w: 0 for w in ranked[:k]} | {w: 1 for w in ranked[-k:]}


def sentence_features(text, vocab, cues):
    toks = tokens(text); grams = ngrams_of(toks)
    hits = [cues[g] for g in grams if g in cues]
    letters = [t for t in toks if re.search(r"[^\W\d_]", t)]
    digit = sum(1 for t in letters if any(ch in DIGIT_LETTERS for ch in t))
    return {"len": len(toks), "oov": (sum(t not in vocab for t in toks) / len(toks)) if toks else 0.0,
            "cue": int(bool(hits)), "vote": (1 if sum(hits) * 2 > len(hits) else 0 if sum(hits) * 2 < len(hits) else None) if hits else None,
            "digit": digit / len(letters) if letters else 0.0,
            "mktn": sum(t in MARKERS["tn"] for t in toks), "mkma": sum(t in MARKERS["ma"] for t in toks),
            "mkdz": sum(t in MARKERS["dz"] for t in toks)}


def model_free(R, data, minority):
    refs = {}
    for script, name in (("ar", "train_ar"), ("lat", "train_gs")):
        if name in data:
            d = data[name]
            refs[script] = {"vocab": {t for x in d["text"] for t in tokens(x)},
                            "cng": char_ngrams(d["text"]), "cues": polarity_cues(d["text"], d["y"])}
    feats = {}
    core = {"train_ar", "val_ar", "ar", "gs", "gu", "tl", "tz", "na", "my", "naar", "myar", "gsmy"}
    for name, d in data.items():
        if name not in core:
            continue
        n = len(d["y"])
        if name == "train_gs":
            continue
        if name == "train_ar":
            key = "train"
        elif name == "val_ar":
            key = "val"
        else:
            key = name
        R.set(f"data/{key}/n", fmt_int(n))
        R.set(f"data/{key}/pos", fmt(100 * d["y"].mean()))
        R.set(f"data/{key}/len", fmt(np.mean([len(tokens(t)) for t in d["text"]])))
        script = "ar" if name in ARABIC_SETS else "lat"
        if script not in refs or name.startswith("train") or name == "val_ar":
            continue
        ref = refs[script]
        F = [sentence_features(t, ref["vocab"], ref["cues"]) for t in d["text"]]
        feats[name] = F
        ntok = sum(f["len"] for f in F) or 1
        R.set(f"data/{name}/oov", fmt(100 * sum(f["oov"] * f["len"] for f in F) / ntok))
        R.set(f"data/{name}/jsd", fmt(100 * jsd(char_ngrams(d["text"]), ref["cng"])))
        R.set(f"data/{name}/cuecov", fmt(100 * np.mean([f["cue"] for f in F])))
        votes = [(f["vote"] if f["vote"] is not None else 1 - minority, yy) for f, yy in zip(F, d["y"]) if f["cue"]]
        if votes:
            R.set(f"data/{name}/cueacc", fmt(100 * np.mean([v == yy for v, yy in votes])))
        for m in ("tn", "ma", "dz"):
            R.set(f"data/{name}/mk{m}", fmt(100 * sum(f["mk" + m] for f in F) / ntok, 2))
        if script == "lat":
            R.set(f"data/{name}/digit", fmt(100 * np.mean([f["digit"] for f in F])))
    return feats


# ------------------------------------------------------------- model-based diagnostics
def load_diag(diag_dir, data):
    out = {}
    if not diag_dir or not os.path.isdir(diag_dir):
        return out
    for system in os.listdir(diag_dir):
        for seed in os.listdir(os.path.join(diag_dir, system)):
            for fn in os.listdir(os.path.join(diag_dir, system, seed)):
                name = fn[:-4]
                rows = read_tsv(os.path.join(diag_dir, system, seed, fn))
                out.setdefault(system, {}).setdefault(seed, {})[name] = rows
    return out


def model_diagnostics(R, diag, preds, data, emb_dir):
    for system, seeds in diag.items():
        acc = defaultdict(list)
        for seed, sets in seeds.items():
            xi = None
            if "val" in sets:
                xi = np.percentile([float(r["maxproto"]) for r in sets["val"]], 5)
            for name, rows in sets.items():
                if name == "val":
                    continue
                alphas = np.array([[float(r[k]) for k in sorted(r) if k.startswith("alpha")] for r in rows])
                for m in range(alphas.shape[1]):
                    acc[(name, f"alpha{m + 1}")].append(alphas[:, m].mean())
                acc[(name, "hgate")].append(float(np.mean(-np.sum(alphas * np.log(np.clip(alphas, 1e-12, 1)), 1))))
                if "plat" in rows[0]:
                    acc[(name, "plat")].append(np.mean([float(r["plat"]) for r in rows]))
                if xi is not None and "maxproto" in rows[0]:
                    acc[(name, "os")].append(100 * np.mean([float(r["maxproto"]) < xi for r in rows]))
                # gate-reliability correlation with single-encoder correctness
                if name in data:
                    y = dict(zip(data[name]["id"], data[name]["y"]))
                    rs = []
                    for m in range(alphas.shape[1]):
                        enc = f"B3-e{m + 1}"
                        s3 = seeds_with(preds, enc, [name])
                        if not s3:
                            continue
                        pr = dict(zip(data[name]["id"], preds[enc][s3[0]][name]["pred"]))
                        corr = np.array([float(pr[r["id"]] == y[r["id"]]) for r in rows])
                        if corr.std() > 0 and alphas[:, m].std() > 0:
                            rs.append(np.corrcoef(alphas[:, m], corr)[0, 1])
                    if rs:
                        acc[(name, "ralpha")].append(float(np.mean(rs)))
        for (name, metric), v in acc.items():
            R.set(f"{system}/{name}/{metric}:m", fmt(np.mean(v), 1 if metric == "os" else 2))
    # proxy A-distance from fused embeddings
    if emb_dir and os.path.isdir(emb_dir):
        from sklearn.linear_model import LogisticRegression
        from sklearn.model_selection import cross_val_score
        for system in os.listdir(emb_dir):
            ref_p = os.path.join(emb_dir, system, "train_gs.npy")
            if not os.path.exists(ref_p):
                continue
            ref = np.load(ref_p)
            for fn in os.listdir(os.path.join(emb_dir, system)):
                name = fn[:-4]
                if name.startswith("train") or name not in data:
                    continue
                X = np.load(os.path.join(emb_dir, system, fn))
                k = min(len(X), len(ref))
                A = ref[RNG.choice(len(ref), k, replace=False)]; B = X[RNG.choice(len(X), k, replace=False)]
                Z = np.vstack([A, B]); t = np.r_[np.zeros(k), np.ones(k)]
                err = 1 - cross_val_score(LogisticRegression(max_iter=2000), Z, t, cv=5).mean()
                R.set(f"{system}/{name}/pad:m", fmt(2 * (1 - 2 * err), 2))


def error_attribution(R, preds, data, feats, diag, system="B12", n_boot=200):
    from sklearn.linear_model import LogisticRegression
    for name in ("tz", "my"):
        if name not in feats:
            continue
        seeds = seeds_with(preds, system, [name])
        if not seeds:
            continue
        votes = np.mean([preds[system][s][name]["pred"] for s in seeds], 0)
        err = ((votes >= 0.5).astype(int) != data[name]["y"]).astype(int)
        F = feats[name]
        cols = ["cue", "oov", "len", "digit", "mktn", "mkma"]
        X = np.array([[f[c] for c in cols] for f in F], float)
        if system in diag:
            seed0 = sorted(diag[system])[0]
            sets = diag[system][seed0]
            if name in sets and "val" in sets and "maxproto" in sets[name][0]:
                xi = np.percentile([float(r["maxproto"]) for r in sets["val"]], 5)
                os_map = {r["id"]: float(float(r["maxproto"]) < xi) for r in sets[name]}
                X = np.c_[X, [os_map.get(i, 0.0) for i in data[name]["id"]]]
                cols = cols + ["os"]
        keep = X.std(0) > 0
        Xs = (X[:, keep] - X[:, keep].mean(0)) / X[:, keep].std(0)
        names = [c for c, k in zip(cols, keep) if k]
        if err.min() == err.max():
            continue
        fit = lambda A, b: LogisticRegression(C=1e6, max_iter=5000).fit(A, b).coef_[0]
        coef = fit(Xs, err)
        boots = []
        for _ in range(n_boot):
            idx = RNG.integers(0, len(err), len(err))
            if err[idx].min() != err[idx].max():
                boots.append(fit(Xs[idx], err[idx]))
        boots = np.array(boots)
        for j, c in enumerate(names):
            lo, hi = np.percentile(np.exp(boots[:, j]), [2.5, 97.5])
            R.set(f"{system}/{name}/or-{c}", fmt_ci(math.exp(coef[j]), lo, hi, 2))


# ------------------------------------------------------------- audits, factorial, figures
def audits(R, audit_dir):
    if not audit_dir or not os.path.isdir(audit_dir):
        return
    for gen in ("gs", "gu"):
        p = os.path.join(audit_dir, f"gen_{gen}.tsv")
        if os.path.exists(p):
            rows = read_tsv(p)
            a = [int(r["ann1"]) for r in rows]; b = [int(r["ann2"]) for r in rows]
            has_final = rows and "final" in rows[0] and all(str(r["final"]).strip() != "" for r in rows)
            final = [int(r["final"]) for r in rows] if has_final else [x * y for x, y in zip(a, b)]
            R.set(f"audit/{gen}/accept", fmt(100 * np.mean(final)))
            R.set(f"audit/{gen}/kappa", fmt(cohen_kappa(a, b), 2))
    for name in ("tz", "my"):
        p = os.path.join(audit_dir, f"errors_{name}.tsv")
        if os.path.exists(p):
            rows = read_tsv(p)
            fin = Counter(r["final"] for r in rows)
            for e in ("E1", "E2", "E3", "E4", "E5", "E6"):
                R.set(f"audit/{name}/{e}", fmt(100 * fin[e] / len(rows)))
            R.set(f"audit/{name}/kappa", fmt(cohen_kappa([r["ann1"] for r in rows], [r["ann2"] for r in rows]), 2))


def factorial(R, preds, data, out):
    systems = [s for s in preds if re.fullmatch(r"fact-[01]{4}", s)]
    if len(systems) < 16:
        return
    rows, ys = [], []
    stages = [s for s in CHAIN if s in data]
    for s in systems:
        bits = [int(b) for b in s[5:]]
        inter = [bits[i] * bits[j] for i in range(4) for j in range(i + 1, 4)]
        for seed in preds[s]:
            for k, st in enumerate(stages):
                if st in preds[s][seed]:
                    dummies = [1.0 if k == j else 0.0 for j in range(1, len(stages))]
                    rows.append([1.0] + bits + inter + dummies)
                    ys.append(100 * f1_macro(data[st]["y"], preds[s][seed][st]["pred"]))
    X = np.array(rows); y = np.array(ys)
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta; dof = len(y) - X.shape[1]
    cov = (resid @ resid / dof) * np.linalg.pinv(X.T @ X)
    se = np.sqrt(np.diag(cov))
    names = ["Fusion", "Proto", "HM", "Cons"]
    terms = names + [f"{names[i]}x{names[j]}" for i in range(4) for j in range(i + 1, 4)]
    table = {}
    for j, t in enumerate(terms, start=1):
        R.set(f"factorial/{t}", fmt_ci(beta[j], beta[j] - 1.96 * se[j], beta[j] + 1.96 * se[j], 2))
        table[t] = [float(beta[j]), float(se[j])]
    json.dump(table, open(os.path.join(out, "factorial.json"), "w"), indent=1)


def write_figures(preds, data, emb_dir, out):
    def stat(system, name):
        v = per_seed_metric(preds, data, system, name, "mf1") if name in data else []
        if not v:
            return float("nan"), float("nan")
        b = boot_mf1(preds, data, system, name, n=200)
        lo, hi = np.percentile(b, [2.5, 97.5])
        return 100 * float(np.mean(v)), 100 * float(hi - lo) / 2
    systems = ["B12", "B10", "B6", "B0", "CEIL"]
    if any(s in preds for s in systems) and "ar" in data:
        with open(os.path.join(out, "chain.dat"), "w") as f:
            f.write("stage " + " ".join(f"{s} {s}err" for s in systems) + "\n")
            for i, name in enumerate(CHAIN):
                vals = []
                for s in systems:
                    m, e = stat(s, name) if (s != "CEIL" or name not in ("gs", "gu")) else (float("nan"),) * 2
                    vals += [m, e]
                f.write(f"{i} " + " ".join("nan" if math.isnan(v) else f"{v:.2f}" for v in vals) + "\n")
    for system, col, tag in (("B6", "p_glob", "glob"), ("B12", "p_var", "var")):
        curves = {}
        for name in HUMAN:
            seeds = seeds_with(preds, system, [name])
            if name in data and seeds and col in preds[system][seeds[0]][name]:
                y = np.concatenate([data[name]["y"]] * len(seeds))
                p = np.concatenate([preds[system][s][name]["pred"] for s in seeds])
                q = np.concatenate([preds[system][s][name][col] for s in seeds])
                curves[name] = reliability(*conf_correct(y, p, q))
        if curves:
            with open(os.path.join(out, f"rel-{system}-{tag}.dat"), "w") as f:
                f.write(" ".join(f"{n}_conf {n}_acc" for n in HUMAN) + "\n")
                for b in range(15):
                    row = []
                    for n in HUMAN:
                        row += list(curves[n][b]) if n in curves and b < len(curves[n]) else [float("nan")] * 2
                    f.write(" ".join("nan" if math.isnan(v) else f"{v:.4f}" for v in row) + "\n")
    sev = [f"rob-tru-s{k}" for k in range(5)]
    if all(s in data for s in sev) and any(s in preds for s in ("B12", "B11", "B6")):
        with open(os.path.join(out, "robcurve-tru.dat"), "w") as f:
            f.write("sev B12 B12err B11 B11err B6 B6err\n")
            for k, name in enumerate(sev):
                row = []
                for s in ("B12", "B11", "B6"):
                    v = per_seed_metric(preds, data, s, name, "acc")
                    row += [100 * np.mean(v), 100 * 1.96 * np.std(v, ddof=1) / math.sqrt(len(v))] if len(v) > 1 else [float("nan")] * 2
                f.write(f"{k} " + " ".join("nan" if math.isnan(v) else f"{v:.2f}" for v in row) + "\n")
    if emb_dir and os.path.isdir(emb_dir):
        try:
            import umap  # umap-learn
        except ImportError:
            print("[info] umap-learn not installed: UMAP figure skipped", file=sys.stderr)
            return
        for system in ("B6", "B12"):
            parts, cls = [], []
            for name in ("ar", "gs", "tl", "my", "myar"):
                p = os.path.join(emb_dir, system, f"{name}.npy")
                if os.path.exists(p) and name in data:
                    X = np.load(p); idx = RNG.choice(len(X), min(400, len(X)), replace=False)
                    parts.append(X[idx]); cls += [f"{name}{data[name]['y'][i]}" for i in idx]
            if parts:
                Y = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=0).fit_transform(np.vstack(parts))
                with open(os.path.join(out, f"umap-{system}.dat"), "w") as f:
                    f.write("x y cls\n")
                    for (a, b), c in zip(Y, cls):
                        f.write(f"{a:.4f} {b:.4f} {c}\n")


def write_cis(preds, data, out):
    """95% bootstrap CIs of macro-F1 for every system on every chain/control set (ci.csv)."""
    with open(os.path.join(out, "ci.csv"), "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["system", "set", "mf1_mean", "ci_low", "ci_high", "n_seeds"])
        for system in sorted(preds):
            if system not in MAIN + ["CEIL", "B12-nopc"]:
                continue
            for name in CHAIN + ["naar", "myar", "gsmy"]:
                v = per_seed_metric(preds, data, system, name, "mf1") if name in data else []
                if not v:
                    continue
                lo, hi = np.percentile(boot_mf1(preds, data, system, name), [2.5, 97.5])
                w.writerow([system, name, f"{100 * np.mean(v):.2f}", f"{100 * lo:.2f}", f"{100 * hi:.2f}", len(v)])


# ------------------------------------------------------------- entry point
def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--data", required=True); ap.add_argument("--preds")
    ap.add_argument("--diag"); ap.add_argument("--emb"); ap.add_argument("--audit")
    ap.add_argument("--manual"); ap.add_argument("--out", default="results")
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)

    data = {}
    for fn in sorted(os.listdir(args.data)):
        if fn.endswith(".tsv"):
            d = load_set(args.data, fn[:-4])
            if d is not None:
                data[fn[:-4]] = d
    minority = int(np.mean(data["train_ar"]["y"]) > 0.5) ^ 1 if "train_ar" in data else 0
    R = Results()
    preds = load_preds(args.preds, data)
    summary = classification_metrics(R, preds, data, minority)
    bold_and_significance(R, preds, data, summary)
    decomposition(R, preds, data)
    feats = model_free(R, data, minority)
    diag = load_diag(args.diag, data)
    model_diagnostics(R, diag, preds, data, args.emb)
    error_attribution(R, preds, data, feats, diag)
    audits(R, args.audit)
    factorial(R, preds, data, args.out)
    write_figures(preds, data, args.emb, args.out)
    write_cis(preds, data, args.out)
    if args.manual:
        for k, v in json.load(open(args.manual, encoding="utf-8")).items():
            R.set(k, v)
    R.write(os.path.join(args.out, "results.tex"))
    json.dump({f"{a}/{b}/{c}": [m, s] for (a, b, c), (m, s, _) in summary.items()},
              open(os.path.join(args.out, "summary.json"), "w"), indent=0)
    print(f"wrote {len(R.vals)} values to {os.path.join(args.out, 'results.tex')}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile download_models.py
#!/usr/bin/env python3
"""Download everything DiaFuse-PRC needs, for offline runs on lab GPU servers.

    python download_models.py --models_dir models --raw_dir raw [--extra bgem3,marbert]

Models are written to <models_dir>/<short name>/ (loadable with from_pretrained(path));
corpora are written to <raw_dir>/<repo>/. Afterwards, on the offline machine:

    export HF_HUB_OFFLINE=1 TRANSFORMERS_OFFLINE=1
    python extract.py --models_dir models ...
"""
import argparse
import io
import os
import subprocess
import urllib.request
import zipfile

MODELS = {
    "xlmr": "FacebookAI/xlm-roberta-large",
    "me5": "intfloat/multilingual-e5-large",
    "labse": "sentence-transformers/LaBSE",
}
EXTRA = {"bgem3": "BAAI/bge-m3", "marbert": "UBC-NLP/MARBERTv2", "qwen": "Qwen/Qwen2.5-7B",
         "qwen_instruct": "Qwen/Qwen2.5-7B-Instruct"}
CORPORA = {  # repository -> branch
    "fbougares/TSAC": "master",
    "chaymafourati/TUNIZI-Sentiment-Analysis-Tunisian-Arabizi-Dataset": "master",
    "SamiaTouileb/Narabizi": "main",
    "MouadJb/MYC": "main",
}
ALLOW = ["*.json", "*.txt", "*.model", "*.safetensors", "sentencepiece*", "tokenizer*", "vocab*",
         "*.py", "1_Pooling/*", "2_Dense/*", "modules.json"]


def download_models(models_dir, names):
    from huggingface_hub import snapshot_download
    for name in names:
        repo = {**MODELS, **EXTRA}[name]
        target = os.path.join(models_dir, name)
        print(f"[models] {repo} -> {target}", flush=True)
        try:
            snapshot_download(repo_id=repo, local_dir=target, allow_patterns=ALLOW)
        except Exception as e:  # some repos ship only pytorch_model.bin
            print(f"  safetensors-only download failed ({e}); retrying with *.bin", flush=True)
            snapshot_download(repo_id=repo, local_dir=target, allow_patterns=ALLOW + ["*.bin"])
        has_weights = any(f.endswith((".safetensors", ".bin")) for f in os.listdir(target))
        if not has_weights:
            snapshot_download(repo_id=repo, local_dir=target, allow_patterns=ALLOW + ["*.bin"])


def download_corpora(raw_dir):
    os.makedirs(raw_dir, exist_ok=True)
    for repo, branch in CORPORA.items():
        name = repo.split("/")[1]
        target = os.path.join(raw_dir, name)
        if os.path.isdir(target):
            print(f"[data] {name} already present"); continue
        print(f"[data] {repo} -> {target}", flush=True)
        try:
            subprocess.run(["git", "clone", "-q", "--depth", "1", f"https://github.com/{repo}.git", target], check=True)
        except Exception:
            for br in (branch, "main", "master"):
                try:
                    data = urllib.request.urlopen(f"https://github.com/{repo}/archive/refs/heads/{br}.zip").read()
                    with zipfile.ZipFile(io.BytesIO(data)) as z:
                        z.extractall(raw_dir)
                    os.rename(os.path.join(raw_dir, f"{name}-{br}"), target)
                    break
                except Exception:
                    continue


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--models_dir", default="models"); ap.add_argument("--raw_dir", default="raw")
    ap.add_argument("--extra", default="", help="comma list among: bgem3, marbert, qwen, qwen_instruct")
    ap.add_argument("--skip_models", action="store_true"); ap.add_argument("--skip_data", action="store_true")
    args = ap.parse_args()
    if not args.skip_models:
        names = list(MODELS) + [e for e in args.extra.split(",") if e]
        download_models(args.models_dir, names)
    if not args.skip_data:
        download_corpora(args.raw_dir)
    print("done")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile gen_llm_perturb.py
#!/usr/bin/env python3
"""Optional: the two never-trained perturbation types of Table 14 (lexical substitution and
paraphrase), generated by an instruction-tuned LLM from the Latin-script view S1 (gs.tsv),
then filtered by a cheap validity check. Writes data/rob-lex.tsv and data/rob-para.tsv
(ids of S1, so that the consistency rate pairs them with gs). Run BEFORE extract.py.
The outputs must be audited by native speakers before being reported (Section 5.3).

    python gen_llm_perturb.py --model models/qwen_instruct   # or Qwen/Qwen2.5-7B-Instruct
"""
import argparse
import csv
import json
import os
import re

import numpy as np
import torch
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import normalize
from transformers import AutoModelForCausalLM, AutoTokenizer

PROMPTS = {
    "lex": ("Here is a social-media comment written in Tunisian Arabic with Latin letters (Arabizi). "
            "Replace two or three words by synonyms that a Tunisian would write in Arabizi, keeping the meaning "
            "and the sentiment unchanged. Answer with the rewritten comment only.\nComment: {x}"),
    "para": ("Here is a social-media comment written in Tunisian Arabic with Latin letters (Arabizi). "
             "Paraphrase it in Tunisian Arabizi, keeping the meaning and the sentiment unchanged. "
             "Answer with the paraphrase only.\nComment: {x}"),
}
MIN_SIM = {"lex": 0.5, "para": 0.2}


def read(path):
    with open(path, encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f, delimiter="\t"))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="Qwen/Qwen2.5-7B-Instruct"); ap.add_argument("--data", default="data")
    ap.add_argument("--batch", type=int, default=16); ap.add_argument("--max_new", type=int, default=64)
    args = ap.parse_args()
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    tok = AutoTokenizer.from_pretrained(args.model); tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(args.model, torch_dtype=torch.float16 if dev == "cuda" else torch.float32).to(dev)
    rows = read(os.path.join(args.data, "gs.tsv")); report = {}
    for kind, prompt in PROMPTS.items():
        outs = []
        for i in range(0, len(rows), args.batch):
            chunk = rows[i:i + args.batch]
            msgs = [[{"role": "user", "content": prompt.format(x=r["text"])}] for r in chunk]
            texts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) if tok.chat_template
                     else m[0]["content"] + "\n" for m in msgs]
            enc = tok(texts, return_tensors="pt", padding=True).to(dev)
            with torch.no_grad():
                gen = model.generate(**enc, max_new_tokens=args.max_new, do_sample=False, pad_token_id=tok.pad_token_id)
            for g in gen[:, enc["input_ids"].shape[1]:]:
                outs.append(tok.decode(g, skip_special_tokens=True).strip().split("\n")[0].strip().strip('"'))
        src = [r["text"] for r in rows]
        vec = CountVectorizer(analyzer="char", ngram_range=(3, 3)).fit(src + outs)
        sim = np.asarray((normalize(vec.transform(src)).multiply(normalize(vec.transform(outs)))).sum(1)).ravel()
        keep = []
        for r, o, s_ in zip(rows, outs, sim):
            ratio = len(o) / max(1, len(r["text"]))
            latin = len(re.findall(r"[A-Za-z]", o)) >= 0.6 * max(1, len(re.findall(r"\w", o)))
            if o and o.lower() != r["text"].lower() and s_ >= MIN_SIM[kind] and 0.5 <= ratio <= 2.0 and latin:
                keep.append((r["id"], o, r["label"]))
        with open(os.path.join(args.data, f"rob-{kind}.tsv"), "w", encoding="utf-8", newline="") as f:
            w = csv.writer(f, delimiter="\t"); w.writerow(["id", "text", "label"]); w.writerows(keep)
        report[kind] = {"generated": len(outs), "kept": len(keep)}
        print(kind, report[kind], flush=True)
    json.dump(report, open(os.path.join(args.data, "llm_perturb_report.json"), "w"), indent=1)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile export_audit.py
#!/usr/bin/env python3
"""Write the annotation sheets of the native-speaker audits (Section 5.3 and Table 10).

    python export_audit.py --data data --runs runs --out audit_todo

audit_todo/gen_gs.tsv, gen_gu.tsv : 200 generator outputs; ann1/ann2 = 1 if a Tunisian writer
                                    could plausibly produce the output, else 0; final = adjudicated.
audit_todo/errors_tz.tsv, errors_my.tsv : 200 B12 errors (majority vote over seeds); ann1/ann2/final
                                    in {E1,...,E6} (definitions in Table 10 of the paper).
Fill the sheets, copy them to audit/, and re-run diafuse_eval.py with --audit audit.
"""
import argparse
import csv
import os

import numpy as np


def read(path):
    with open(path, encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f, delimiter="\t"))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data"); ap.add_argument("--runs", default="runs")
    ap.add_argument("--out", default="audit_todo"); ap.add_argument("--n", type=int, default=200)
    args = ap.parse_args(); os.makedirs(args.out, exist_ok=True); rng = np.random.default_rng(7)
    src = {r["id"]: r["text"] for r in read(os.path.join(args.data, "ar.tsv"))}
    for g in ("gs", "gu"):
        rows = read(os.path.join(args.data, f"{g}.tsv"))
        pick = rng.choice(len(rows), min(args.n, len(rows)), replace=False)
        with open(os.path.join(args.out, f"gen_{g}.tsv"), "w", encoding="utf-8", newline="") as f:
            w = csv.writer(f, delimiter="\t"); w.writerow(["id", "source_arabic", "output", "ann1", "ann2", "final"])
            for i in pick:
                w.writerow([rows[i]["id"], src.get(rows[i]["id"], ""), rows[i]["text"], "", "", ""])
    for s in ("tz", "my"):
        rows = read(os.path.join(args.data, f"{s}.tsv")); y = np.array([int(r["label"]) for r in rows])
        sd = os.path.join(args.runs, "preds", "B12")
        seeds = [d for d in os.listdir(sd) if os.path.exists(os.path.join(sd, d, f"{s}.tsv"))] if os.path.isdir(sd) else []
        if not seeds:
            print(f"no B12 predictions for {s}; run diafuse_run.py first"); continue
        votes = np.mean([[int(r["pred"]) for r in read(os.path.join(sd, d, f"{s}.tsv"))] for d in seeds], 0)
        pred = (votes >= 0.5).astype(int); err = np.flatnonzero(pred != y)
        pick = rng.choice(err, min(args.n, len(err)), replace=False)
        with open(os.path.join(args.out, f"errors_{s}.tsv"), "w", encoding="utf-8", newline="") as f:
            w = csv.writer(f, delimiter="\t"); w.writerow(["id", "text", "gold", "pred", "ann1", "ann2", "final"])
            for i in pick:
                w.writerow([rows[i]["id"], rows[i]["text"], y[i], pred[i], "", "", ""])
    print("annotation sheets written to", args.out)


if __name__ == "__main__":
    main()

## Étape 1 — téléchargement des modèles et des corpus

In [ ]:
extra = [e for e, on in (("marbert", RUN_M4), ("qwen", RUN_QWEN_COST), ("qwen_instruct", RUN_LLM_PERTURB)) if on]
run(f"python download_models.py --models_dir models --raw_dir raw" + (f" --extra {','.join(extra)}" if extra else ""))

## Étape 2 — reconstruction des splits (Section 5.2) et générateurs

In [ ]:
if not os.path.exists("data/prep_report.json"):
    env = {}
    if MADAR_DIR:
        env["MADAR_DIR"] = MADAR_DIR
    if USE_OWN_SPLITS:  # train_ar, val_ar, ar, tl, tz, na, naar, my, myar (.tsv: id, text, label)
        env["OWN_SPLITS"] = USE_OWN_SPLITS
    run("python prep_data.py", env=env)
rep = json.load(open("data/prep_report.json")) if os.path.exists("data/prep_report.json") else {}
paper = {"val_ar": 547, "tl": 1020, "tz": 1737, "na": 696, "my": 1938}
print(f"{'set':8s} {'this run':>9s} {'your pipeline':>14s}")
for k in ("train_ar", "val_ar", "ar", "tl", "tz", "na", "my", "myar"):
    print(f"{k:8s} {rep.get('final/' + k, '?'):>9} {paper.get(k, '-'):>14}")

## Étape 3 (optionnelle) — perturbations LLM (substitution lexicale, paraphrase)

In [ ]:
if RUN_LLM_PERTURB and not os.path.exists("data/rob-lex.tsv"):
    run("python gen_llm_perturb.py --model models/qwen_instruct --data data")

## Étape 4 — extraction des vecteurs par couche (Eq. 1)

In [ ]:
encs = ENCODERS + (",marbert" if RUN_M4 else "")
run(f"python extract.py --models_dir models --data data --out feats --encoders {encs} --layer_stride {LAYER_STRIDE} --batch 128")
if RUN_QWEN_COST:
    run(f"python extract.py --models_dir models --data data --out feats --encoders qwen --cost_only --batch 32")

## Étape 5 — DiaFuse-PRC, baselines, ablations, déséquilibre contrôlé, factoriel, plafond CEIL

In [ ]:
encs = ENCODERS + (",marbert" if RUN_M4 else "")
run(f"python diafuse_run.py --data data --feats feats --out runs --mode {MODE} --encoders {encs}"
    + (f" --skip {SKIP_GROUPS}" if SKIP_GROUPS else ""))

## Étape 6 — baselines fine-tunées B1 et B11

In [ ]:
if RUN_FINETUNE:
    for system in ("B1", "B11"):
        run(f"python finetune.py --system {system} --model models/xlmr --data data --out runs --seeds {FINETUNE_SEEDS}")

## Étape 7 — feuilles d'annotation des audits humains

Les fichiers `audit_todo/*.tsv` doivent être remplis par deux locuteurs natifs (colonnes `ann1`, `ann2`, `final`),
puis copiés dans `audit/`. Relancez ensuite l'étape 8 : les Tableaux 10 et l'acceptabilité des générateurs
seront alors calculés. Sans ces fichiers, ces cellules restent vides dans l'article.

In [ ]:
run("python export_audit.py --data data --runs runs --out audit_todo")
if USE_DRIVE:
    shutil.copytree("audit_todo", f"{DRIVE_DIR}/audit_todo", dirs_exist_ok=True)
print(sorted(os.listdir("audit_todo")))

## Étape 8 — évaluation complète et `results/results.tex`

In [ ]:
manual = {}
for p in ("data/manual_counts.json", "runs/manual.json"):
    if os.path.exists(p):
        manual.update(json.load(open(p)))
json.dump(manual, open("results/manual_all.json", "w"), indent=1)
has_audit = any(f.endswith(".tsv") for f in os.listdir("audit"))
run("python diafuse_eval.py --data data --preds runs/preds --diag runs/diag --emb runs/emb "
    "--manual results/manual_all.json --out results" + (" --audit audit" if has_audit else ""))

## Étape 9 — archive à renvoyer

In [ ]:
import torch, transformers, platform
env = {"mode": MODE, "encoders": ENCODERS, "layer_stride": LAYER_STRIDE, "finetune_seeds": FINETUNE_SEEDS,
       "llm_perturb": RUN_LLM_PERTURB, "qwen_cost": RUN_QWEN_COST, "m4": RUN_M4, "madar": bool(MADAR_DIR),
       "own_splits": bool(USE_OWN_SPLITS), "torch": torch.__version__, "transformers": transformers.__version__,
       "python": platform.python_version(),
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
       "date": time.strftime("%Y-%m-%d %H:%M")}
json.dump(env, open("results/environment.json", "w"), indent=1)
bundle = "results_bundle.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob("results/*"):
        z.write(f)
    for f in ("data/prep_report.json", "data/manual_counts.json", "data/llm_perturb_report.json",
              "runs/dev_scores.json", "runs/b12_hparams.json", "runs/manual.json"):
        if os.path.exists(f):
            z.write(f)
print("bundle:", bundle, os.path.getsize(bundle) // 1024, "KB")
try:
    from google.colab import files
    files.download("results/results.tex"); files.download(bundle)
except Exception as e:
    print("download from the file browser:", e)

## Notes d'implémentation (à reporter dans l'article avec les résultats)

1. **Splits** reconstruits depuis les releases publiques (Section 5.2), sauf si `USE_OWN_SPLITS` est utilisé ; effectifs dans `prep_report.json`.
2. **Couches** : une couche sur deux est mise en cache (0, 2, …, L), comme envisagé à l'annexe H.
3. **Cibles contrastives** : tirées d'une banque des embeddings d'entraînement rafraîchie à chaque époque (pools de 256 par cellule), au lieu d'un tirage direct dans tout le jeu.
4. **Échantillons synthétiques** : jusqu'à 5 positifs et 50 négatifs mixés par ancre, selon $r_t = 0.5 + 0.5 t/T$.
5. **Validité** : q = 1 pour toutes les familles (TR, INF, NOISE déterministes ; CS par lexique arabe→français) ; pas de filtre appris, donc pas de variante abl2-q1.
6. **B10** : réimplémentation SharpReCL-style (logit adjustment, prototypes de classe, simple-sampling, hard-mixup par similarité au prototype), pas le code publié.
7. **B5** : couche et pooling choisis sur le dev par une régression logistique, puis tête MLP.
8. **Pooling par défaut** : moyenne de la dernière couche (XLM-R, mE5 avec le préfixe `query: `), token [CLS] pour LaBSE.
9. **B1** : tête sur `<s>`, 3 époques, lr 1e-5 ; **B11** : pooling attentionnel, CE sur les deux vues + consistance ℓ2 (λ = 0.5).
10. **Perturbations LLM** (optionnelles) : Qwen2.5-7B-Instruct, filtre de validité simple (similarité de trigrammes de caractères, ratio de longueur, script latin).
11. **Non exécuté** : B12′ (Setting B, LoRA).